# RLCoder: learned phase + log fidelity
768 qubits, original tanh theta; phi = pi + pi*tanh(Linear(h)). No compression.
16 epochs; generate validation every 2 epochs; macro EM selects best, ES breaks ties.
Selection set: fixed 200 samples/language from CrossCodeEval TEST, original ground truth; saved in validation_crosscodeeval.jsonl. Best saved in best_crosscodeeval/.
Start SMOKE=True. Smoke is a separate 1-epoch run without validation; start fresh for full training.
Resume interrupted full run: SMOKE=False; set RESUME to latest.pt from previous output. Config, RNG, optimizer, validation and best weights restored; switching from old synthetic validation resets best only.
A 16-epoch run may span Kaggle sessions. Save Version, attach prior output, and resume. Max runtime 660 minutes/session.
Do not resume fixed-phase checkpoints into learned-phase architecture. This is a QIEPSM-inspired variant.


In [ ]:
SMOKE = True
SAMPLES_PER_EPOCH = 2000
EPOCHS = 16
LOG_EVERY_UPDATES = 50
EVAL_EPOCHS = list(range(2, 17, 2))
BENCHMARK_LIMIT = 0  # validation uses all saved rows
VALID_PER_LANGUAGE = 200
SCORER = "qiepsm_learned_phase"  # QIEPSM encoding, modified log objective
LOSS = "hard"  # upstream-style CE; optional soft / hybrid
SCORE_SCALE = 0.1  # scorer logit scale
SEED = 123
RESUME = ""  # e.g. /kaggle/input/previous-run/latest.pt; restores training config
GENERATOR_BATCH_SIZE = 2
ENCODE_BATCH_SIZE = 16
TRAIN_BATCH_SIZE = 4
ACCUMULATION_STEPS = 4  # effective retriever batch = 16
MAX_RUNTIME_MINUTES = 660  # stop gracefully around 11h; set 0 to disable


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.44,<5", "accelerate>=0.33", "rank-bm25>=0.2.2",
    "pandas>=2", "pyarrow>=15", "huggingface-hub>=0.24"], check=True)


In [ ]:
import torch, transformers, pandas, pyarrow
print("Python:", sys.version)
print("Torch:", torch.__version__, "CUDA build:", torch.version.cuda,
      "CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is not enabled. Open Notebook options > Accelerator and select a GPU.")
print("GPU:", torch.cuda.get_device_name(0),
      "VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download
root = Path("/kaggle/working/Data4RLCoder")
snapshot_download("nov3630/Data4RLCoder", repo_type="dataset", local_dir=root,
                  allow_patterns=["*github_repos*train.parquet"])
found = list(root.rglob("github_repos/python/train.parquet"))
if len(found) != 1:
    raise RuntimeError(f"Expected one Python parquet, found {found}")
DATA = found[0].parent.parent.parent
assert (DATA / "github_repos/java/train.parquet").exists()


In [ ]:
ENGINE = Path("/kaggle/working/online_engine")
ENGINE.mkdir(exist_ok=True)
(ENGINE / 'controlled_comparison.py').write_text('#!/usr/bin/env python\n"""Controlled cosine-vs-quantum-fidelity experiment for RLCoder.\n\nThe script deliberately separates reward-data preparation from retriever\ntraining.  Both scorers therefore see the exact same queries, candidates, and\nDeepSeek-Coder losses.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport math\nimport os\nimport random\nimport re\nimport time\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom rank_bm25 import BM25Okapi\nfrom torch.optim import AdamW\nfrom torch.utils.data import DataLoader, Dataset\nfrom tqdm.auto import tqdm\nfrom transformers import (\n    AutoModel,\n    AutoModelForCausalLM,\n    AutoTokenizer,\n    get_linear_schedule_with_warmup,\n)\n\n\nNULL_CONTEXT = "Don\'t need cross file context for completion"\n\n\n@dataclass\nclass Candidate:\n    file_path: str\n    description: str\n    code: str\n    language: str\n    is_null: bool = False\n\n    def render(self) -> str:\n        marker = "#" if self.language == "python" else "//"\n        if self.is_null:\n            return ""\n        description = "\\n".join(\n            f"{marker} {line}" for line in self.description.splitlines() if line.strip()\n        )\n        code = "\\n".join(f"{marker} {line}" for line in self.code.splitlines() if line.strip())\n        return f"{description}\\n{code}".strip()\n\n\n@dataclass\nclass CompletionExample:\n    task_id: str\n    file_path: str\n    left_context: str\n    target_code: str\n    language: str\n    related_files: list[tuple[str, str]]\n\n\ndef seed_everything(seed: int) -> None:\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef resolve_data_root(data_root: str) -> Path:\n    requested = Path(data_root)\n    candidates = [\n        requested,\n        requested / "data",\n        Path("/kaggle/input/data4rlcoder/data"),\n        Path("/kaggle/input/data4rlcoder"),\n        Path("data"),\n        Path("RLCoder/data"),\n    ]\n    for candidate in candidates:\n        if (candidate / "github_repos/python/train.parquet").exists():\n            return candidate.resolve()\n    searched = "\\n  - ".join(str(path) for path in candidates)\n    raise FileNotFoundError(\n        "Could not find github_repos/python/train.parquet. Searched:\\n  - " + searched\n    )\n\n\ndef load_repository_groups(data_root: Path) -> tuple[list, list]:\n    train_groups: list[tuple[list[tuple[str, str]], str]] = []\n    valid_groups: list[tuple[list[tuple[str, str]], str]] = []\n    for language in ("python", "java"):\n        frame = pd.read_parquet(data_root / f"github_repos/{language}/train.parquet")\n        groups: list[list[tuple[str, str]]] = []\n        current: list[tuple[str, str]] = []\n        for path, content, first in frame[["path", "content", "first"]].itertuples(index=False, name=None):\n            if first and current:\n                if len(current) > 1:\n                    groups.append(current)\n                current = []\n            current.append((str(path), str(content)))\n        if len(current) > 1:\n            groups.append(current)\n        train_groups.extend((group, language) for group in groups[:2000])\n        valid_groups.extend((group, language) for group in groups[2000:2200])\n    return train_groups, valid_groups\n\n\ndef make_examples(groups: Sequence, count: int, seed: int, split: str) -> list[CompletionExample]:\n    rng = random.Random(seed)\n    order = list(range(len(groups)))\n    rng.shuffle(order)\n    examples: list[CompletionExample] = []\n    attempts = 0\n    max_attempts = max(count * 100, 1000)\n    while len(examples) < count and attempts < max_attempts:\n        group, language = groups[order[attempts % len(order)]]\n        attempts += 1\n        selected_index = rng.randrange(1, len(group))\n        selected_path, selected_content = group[selected_index]\n        tokens = selected_content.split()\n        if len(tokens) < 120:\n            continue\n        cut = int(len(tokens) * rng.uniform(0.2, 0.8))\n        target_length = rng.randint(32, 64)\n        left = " ".join(tokens[:cut])\n        target = " ".join(tokens[cut : cut + target_length])\n        if len(left.split()) <= 80 or len(target.split()) <= 8:\n            continue\n        related = [(path, content) for i, (path, content) in enumerate(group) if i != selected_index]\n        examples.append(\n            CompletionExample(\n                task_id=f"{split}-{len(examples)}",\n                file_path=selected_path,\n                left_context=left,\n                target_code=target,\n                language=language,\n                related_files=related,\n            )\n        )\n    if len(examples) != count:\n        raise RuntimeError(f"Built only {len(examples)}/{count} examples after {attempts} attempts")\n    return examples\n\n\ndef chunk_related_files(example: CompletionExample, block_lines: int) -> list[Candidate]:\n    candidates: list[Candidate] = []\n    for path, content in example.related_files:\n        lines = [line for line in content.splitlines() if line.strip()][:5000]\n        for start in range(0, len(lines), block_lines):\n            block = lines[start : start + block_lines]\n            if not block:\n                continue\n            end = start + len(block) - 1\n            candidates.append(\n                Candidate(\n                    file_path=path,\n                    description=f"file path: {path}\\nlines: {start}-{end}",\n                    code="\\n".join(block),\n                    language=example.language,\n                )\n            )\n    return candidates\n\n\ndef lexical_tokens(text: str) -> list[str]:\n    return re.findall(r"[A-Za-z_][A-Za-z_0-9]*|\\d+|\\S", text.lower())\n\n\ndef bm25_candidates(example: CompletionExample, topk: int, block_lines: int) -> list[Candidate]:\n    blocks = chunk_related_files(example, block_lines)\n    if not blocks:\n        return []\n    corpus = [lexical_tokens(block.code) for block in blocks]\n    index = BM25Okapi(corpus)\n    query = "\\n".join(line for line in example.left_context.splitlines() if line.strip())[-8000:]\n    scores = index.get_scores(lexical_tokens(query))\n    order = np.argsort(scores)[::-1][:topk]\n    return [blocks[int(i)] for i in order]\n\n\ndef rlcoder_tokenize(text: str, tokenizer, max_length: int, is_query: bool) -> list[int]:\n    tokens = tokenizer.tokenize(text)\n    room = max_length - 4\n    tokens = tokens[-room:] if is_query else tokens[:room]\n    tokens = [tokenizer.cls_token, "<encoder-only>", tokenizer.sep_token] + tokens + [tokenizer.sep_token]\n    ids = tokenizer.convert_tokens_to_ids(tokens)\n    return ids + [tokenizer.pad_token_id] * (max_length - len(ids))\n\n\ndef mean_pool(last_hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    weights = mask.unsqueeze(-1).to(last_hidden.dtype)\n    return (last_hidden * weights).sum(1) / weights.sum(1).clamp_min(1)\n\n\n@torch.inference_mode()\ndef encode_texts(\n    model,\n    tokenizer,\n    texts: Sequence[str],\n    max_length: int,\n    batch_size: int,\n    device,\n    *,\n    is_query: bool = False,\n) -> torch.Tensor:\n    output: list[torch.Tensor] = []\n    for start in range(0, len(texts), batch_size):\n        ids = [\n            rlcoder_tokenize(text, tokenizer, max_length, is_query)\n            for text in texts[start : start + batch_size]\n        ]\n        input_ids = torch.tensor(ids, dtype=torch.long, device=device)\n        mask = input_ids.ne(tokenizer.pad_token_id)\n        hidden = model(input_ids=input_ids, attention_mask=mask).last_hidden_state\n        output.append(F.normalize(mean_pool(hidden, mask), p=2, dim=-1).cpu())\n    return torch.cat(output)\n\n\ndef build_fixed_candidate_pools(\n    examples: Sequence[CompletionExample], args, device\n) -> list[tuple[CompletionExample, list[Candidate]]]:\n    tokenizer = AutoTokenizer.from_pretrained(args.retriever_model)\n    model = AutoModel.from_pretrained(args.retriever_model).to(device).eval()\n    pools: list[tuple[CompletionExample, list[Candidate]]] = []\n    for example in tqdm(examples, desc="Freezing candidate pools"):\n        lexical = bm25_candidates(example, args.bm25_topk, args.block_lines)\n        if len(lexical) < args.candidate_count - 1:\n            continue\n        query = "\\n".join(\n            line for line in example.left_context.splitlines() if line.strip()\n        )[-8000:]\n        q = encode_texts(\n            model,\n            tokenizer,\n            [query],\n            args.query_length,\n            1,\n            device,\n            is_query=True,\n        )[0]\n        docs = encode_texts(\n            model,\n            tokenizer,\n            [candidate.render() for candidate in lexical],\n            args.candidate_length,\n            args.retriever_batch_size,\n            device,\n        )\n        scores = docs @ q\n        keep = torch.topk(scores, k=args.candidate_count - 1).indices.tolist()\n        selected = [lexical[i] for i in keep]\n        selected.append(\n            Candidate("", NULL_CONTEXT, "", example.language, is_null=True)\n        )\n        pools.append((example, selected))\n    del model\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return pools\n\n\ndef generator_prompt(example: CompletionExample, candidate: Candidate, tokenizer, args) -> list[int]:\n    cross_ids = tokenizer.encode(candidate.render(), add_special_tokens=False)[: args.generator_crossfile_length]\n    marker = "#" if example.language == "python" else "//"\n    path_ids = tokenizer.encode(\n        f"\\n\\n{marker} file path: {example.file_path}\\n\\n", add_special_tokens=False\n    )\n    allowed = max(args.generator_context_length - len(cross_ids) - len(path_ids) - 10, 1)\n    in_file = tokenizer.encode(example.left_context, add_special_tokens=False)[-allowed:]\n    return (cross_ids + path_ids + in_file)[-args.generator_context_length :]\n\n\n@torch.inference_mode()\ndef score_generator_losses(pools, args, device) -> list[dict]:\n    tokenizer = AutoTokenizer.from_pretrained(args.generator_model)\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token_id = tokenizer.eos_token_id\n    dtype = torch.float16 if device.type == "cuda" else torch.float32\n    model = AutoModelForCausalLM.from_pretrained(args.generator_model, torch_dtype=dtype).to(device).eval()\n\n    flat: list[tuple[int, CompletionExample, Candidate]] = []\n    for example_index, (example, candidates) in enumerate(pools):\n        flat.extend((example_index, example, candidate) for candidate in candidates)\n    losses = [0.0] * len(flat)\n    for start in tqdm(range(0, len(flat), args.generator_batch_size), desc="Freezing PPL rewards"):\n        batch = flat[start : start + args.generator_batch_size]\n        sequences: list[list[int]] = []\n        labels: list[list[int]] = []\n        for _, example, candidate in batch:\n            prompt_ids = generator_prompt(example, candidate, tokenizer, args)\n            target_ids = tokenizer.encode(example.target_code, add_special_tokens=False)[: args.target_length]\n            sequences.append(prompt_ids + target_ids)\n            labels.append([-100] * len(prompt_ids) + target_ids)\n        width = max(len(sequence) for sequence in sequences)\n        padded_ids, padded_labels, masks = [], [], []\n        for sequence, label in zip(sequences, labels):\n            padding = width - len(sequence)\n            padded_ids.append([tokenizer.pad_token_id] * padding + sequence)\n            padded_labels.append([-100] * padding + label)\n            masks.append([0] * padding + [1] * len(sequence))\n        input_ids = torch.tensor(padded_ids, dtype=torch.long, device=device)\n        label_tensor = torch.tensor(padded_labels, dtype=torch.long, device=device)\n        attention_mask = torch.tensor(masks, dtype=torch.long, device=device)\n        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1].float()\n        shifted = label_tensor[:, 1:]\n        token_loss = F.cross_entropy(\n            logits.reshape(-1, logits.shape[-1]),\n            shifted.reshape(-1),\n            reduction="none",\n            ignore_index=-100,\n        ).view(shifted.shape)\n        per_sample = token_loss.sum(-1) / shifted.ne(-100).sum(-1).clamp_min(1)\n        losses[start : start + len(batch)] = per_sample.cpu().tolist()\n\n    rows: list[dict] = []\n    cursor = 0\n    for split_and_pool in pools:\n        example, candidates = split_and_pool\n        candidate_losses = losses[cursor : cursor + len(candidates)]\n        cursor += len(candidates)\n        rows.append(\n            {\n                "task_id": example.task_id,\n                "query": "\\n".join(\n                    line for line in example.left_context.splitlines() if line.strip()\n                )[-8000:],\n                "candidates": [candidate.render() for candidate in candidates],\n                "candidate_meta": [asdict(candidate) for candidate in candidates],\n                "generator_losses": candidate_losses,\n                "label": int(np.argmin(candidate_losses)),\n            }\n        )\n    del model\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return rows\n\n\ndef write_jsonl(path: Path, rows: Iterable[dict]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        for row in rows:\n            handle.write(json.dumps(row, ensure_ascii=False) + "\\n")\n\n\ndef read_jsonl(path: Path) -> list[dict]:\n    with path.open(encoding="utf-8") as handle:\n        return [json.loads(line) for line in handle if line.strip()]\n\n\ndef prepare_reward_cache(args, device) -> None:\n    data_root = resolve_data_root(args.data_root)\n    train_groups, valid_groups = load_repository_groups(data_root)\n    splits = {\n        "train": make_examples(train_groups, args.train_examples, args.seed, "train"),\n        "valid": make_examples(valid_groups, args.valid_examples, args.seed + 1, "valid"),\n    }\n    cache_dir = Path(args.cache_dir)\n    manifest = {\n        "seed": args.seed,\n        "retriever_model": args.retriever_model,\n        "generator_model": args.generator_model,\n        "candidate_count": args.candidate_count,\n        "bm25_topk": args.bm25_topk,\n        "train_examples_requested": args.train_examples,\n        "valid_examples_requested": args.valid_examples,\n    }\n    for split, examples in splits.items():\n        pools = build_fixed_candidate_pools(examples, args, device)\n        if len(pools) != len(examples):\n            print(f"Warning: kept {len(pools)}/{len(examples)} {split} examples with enough candidates")\n        rows = score_generator_losses(pools, args, device)\n        write_jsonl(cache_dir / f"{split}.jsonl", rows)\n        manifest[f"{split}_examples_written"] = len(rows)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    (cache_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    print(f"Frozen reward cache written to {cache_dir.resolve()}")\n\n\nclass FrozenRewardDataset(Dataset):\n    def __init__(self, rows: Sequence[dict], tokenizer, query_length: int, candidate_length: int):\n        self.rows = rows\n        self.tokenizer = tokenizer\n        self.query_length = query_length\n        self.candidate_length = candidate_length\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def __getitem__(self, index: int):\n        row = self.rows[index]\n        query = rlcoder_tokenize(row["query"], self.tokenizer, self.query_length, True)\n        candidates = [\n            rlcoder_tokenize(text, self.tokenizer, self.candidate_length, False)\n            for text in row["candidates"]\n        ]\n        return {\n            "query_ids": torch.tensor(query, dtype=torch.long),\n            "candidate_ids": torch.tensor(candidates, dtype=torch.long),\n            "label": torch.tensor(row["label"], dtype=torch.long),\n            "generator_losses": torch.tensor(row["generator_losses"], dtype=torch.float),\n        }\n\n\nclass QIEPSMFidelity(nn.Module):\n    """Original uncompressed QiBERT encoding and raw product fidelity.\n\n    No learned projection, normalization, learned phase, clamp or log score.\n    Source: qiepsm/compression experiments/bert_base/qi bert base trained/qi-bert.py\n    and its utils/utils.py fidelity(). One qubit per encoder dimension.\n    """\n\n    def states(self, embeddings: torch.Tensor) -> torch.Tensor:\n        theta = torch.tanh(embeddings) * torch.pi / 2 + torch.pi / 2\n        # Retain the original phase expression (approximately pi).\n        phase = torch.ones_like(theta) * (2 * torch.pi - 1e-9) / 2\n        return torch.stack((torch.cos(theta / 2),\n                            torch.sin(theta / 2) * torch.exp(1j * phase)), dim=-1)\n\n    def pairwise_fidelity(self, query: torch.Tensor, docs: torch.Tensor) -> torch.Tensor:\n        q = self.states(query).unsqueeze(1)\n        d = self.states(docs)\n        overlap = (q.conj() * d).sum(dim=-1)\n        # Diagnostic only: sum logs before product underflow, using the same states.\n        self.last_log_fidelity = 2 * overlap.detach().abs().log().sum(dim=-1)\n        return overlap.prod(dim=-1).abs().square()\n\n    def pairwise_log_fidelity(self, query, docs):\n        q, d = self.states(query).unsqueeze(1), self.states(docs)\n        magnitude = (q.conj()*d).sum(-1).abs()\n        # Only guard exact/numerically unrepresentable zero, not a 1e-7 floor.\n        value = 2 * magnitude.clamp_min(torch.finfo(magnitude.dtype).tiny).log().sum(-1)\n        self.last_log_fidelity = value.detach()\n        return value\n\n\nclass QIEPSMLearnedPhase(QIEPSMFidelity):\n    """Uncompressed theta encoding with an input-dependent trainable phase."""\n    def __init__(self, hidden_size):\n        super().__init__()\n        self.phase_projection = nn.Linear(hidden_size, hidden_size)\n        nn.init.normal_(self.phase_projection.weight, std=0.01 / math.sqrt(hidden_size))\n        nn.init.zeros_(self.phase_projection.bias)\n\n    def states(self, embeddings):\n        theta = torch.tanh(embeddings) * torch.pi / 2 + torch.pi / 2\n        phase = torch.pi + torch.pi * torch.tanh(self.phase_projection(embeddings))\n        return torch.stack((torch.cos(theta / 2),\n                            torch.sin(theta / 2) * torch.exp(1j * phase)), -1)\n\n\nclass QuantumFidelityHead(nn.Module):\n    """QIEPSM-inspired product-state projection with stable log fidelity."""\n\n    def __init__(self, hidden_size: int, n_qubits: int, dropout: float):\n        super().__init__()\n        self.n_qubits = n_qubits\n        self.projection = nn.Sequential(\n            nn.LayerNorm(hidden_size),\n            nn.Dropout(dropout),\n            nn.Linear(hidden_size, 2 * n_qubits),\n        )\n\n    def states(self, embeddings: torch.Tensor) -> torch.Tensor:\n        angles = self.projection(embeddings)\n        theta_raw, phi_raw = angles.chunk(2, dim=-1)\n        theta = torch.sigmoid(theta_raw) * math.pi\n        phi = torch.tanh(phi_raw) * math.pi\n        amplitude_0 = torch.cos(theta / 2).to(torch.complex64)\n        amplitude_1 = torch.sin(theta / 2).to(torch.complex64) * torch.exp(1j * phi.float())\n        return torch.stack((amplitude_0, amplitude_1), dim=-1)\n\n    def pairwise_log_fidelity(self, query: torch.Tensor, docs: torch.Tensor) -> torch.Tensor:\n        q_state = self.states(query).unsqueeze(1)\n        d_state = self.states(docs)\n        overlap = (q_state.conj() * d_state).sum(dim=-1).abs().square()\n        # mean(log F_i) ranks like product(F_i), without numerical underflow.\n        return torch.log(overlap.clamp_min(1e-7)).mean(dim=-1)\n\n\nclass ControlledRetriever(nn.Module):\n    def __init__(self, model_name: str, scorer: str, n_qubits: int, dropout: float, temperature: float):\n        super().__init__()\n        self.encoder = AutoModel.from_pretrained(model_name)\n        self.scorer = scorer\n        self.temperature = temperature\n        if scorer == "qiepsm_learned_phase":\n            self.quantum_head = QIEPSMLearnedPhase(self.encoder.config.hidden_size)\n        elif scorer in ("qiepsm_fidelity", "qiepsm_log_fidelity"):\n            if scorer == "qiepsm_fidelity" and temperature != 20.:\n                raise ValueError(\'QIEPSM fidelity requires the original score scale 20.\')\n            self.quantum_head = QIEPSMFidelity()\n        elif scorer == "quantum_fidelity":\n            self.quantum_head = QuantumFidelityHead(\n                self.encoder.config.hidden_size, n_qubits, dropout\n            )\n        elif scorer != "cosine":\n            raise ValueError(f"Unknown scorer: {scorer}")\n\n    def embed(self, input_ids: torch.Tensor, pad_token_id: int) -> torch.Tensor:\n        mask = input_ids.ne(pad_token_id)\n        hidden = self.encoder(input_ids=input_ids, attention_mask=mask).last_hidden_state\n        return mean_pool(hidden, mask)\n\n    def forward(self, query_ids: torch.Tensor, candidate_ids: torch.Tensor, pad_token_id: int):\n        batch, candidate_count, length = candidate_ids.shape\n        query = self.embed(query_ids, pad_token_id)\n        docs = self.embed(candidate_ids.reshape(-1, length), pad_token_id).view(batch, candidate_count, -1)\n        if self.scorer == "cosine":\n            return F.cosine_similarity(query.unsqueeze(1), docs, dim=-1) * self.temperature\n        if self.scorer == "qiepsm_fidelity":\n            return self.quantum_head.pairwise_fidelity(query.float(), docs.float()) * self.temperature\n        return self.quantum_head.pairwise_log_fidelity(query, docs) * self.temperature\n\n\ndef ranking_metrics(logits: torch.Tensor, labels: torch.Tensor, generator_losses: torch.Tensor) -> dict:\n    order = logits.argsort(dim=-1, descending=True)\n    ranks = order.eq(labels.unsqueeze(1)).float().argmax(dim=1) + 1\n    selected = logits.argmax(dim=-1)\n    selected_loss = generator_losses.gather(1, selected.unsqueeze(1)).squeeze(1)\n    oracle_loss = generator_losses.min(dim=1).values\n    return {\n        "correct": selected.eq(labels).sum().item(),\n        "mrr_sum": (1.0 / ranks.float()).sum().item(),\n        "ndcg_sum": (1.0 / torch.log2(ranks.float() + 1)).sum().item(),\n        "regret_sum": (selected_loss - oracle_loss).sum().item(),\n        "count": labels.numel(),\n    }\n\n\n@torch.inference_mode()\ndef evaluate(model, loader, pad_token_id: int, device) -> dict:\n    model.eval()\n    totals = {"loss_sum": 0.0, "correct": 0, "mrr_sum": 0.0, "ndcg_sum": 0.0, "regret_sum": 0.0, "count": 0}\n    for batch in loader:\n        batch = {key: value.to(device) for key, value in batch.items()}\n        logits = model(batch["query_ids"], batch["candidate_ids"], pad_token_id)\n        loss = F.cross_entropy(logits, batch["label"], reduction="sum")\n        metrics = ranking_metrics(logits, batch["label"], batch["generator_losses"])\n        totals["loss_sum"] += loss.item()\n        for key, value in metrics.items():\n            totals[key] += value\n    count = max(totals.pop("count"), 1)\n    return {\n        "loss": totals["loss_sum"] / count,\n        "top1_accuracy": totals["correct"] / count,\n        "mrr": totals["mrr_sum"] / count,\n        "ndcg": totals["ndcg_sum"] / count,\n        "generator_loss_regret": totals["regret_sum"] / count,\n    }\n\n\ndef save_checkpoint(model, tokenizer, output_dir: Path, epoch: int, args, optimizer, scheduler) -> None:\n    checkpoint = output_dir / f"checkpoint-epoch-{epoch}"\n    checkpoint.mkdir(parents=True, exist_ok=True)\n    model.encoder.save_pretrained(checkpoint / "encoder")\n    tokenizer.save_pretrained(checkpoint / "encoder")\n    if model.scorer == "quantum_fidelity":\n        torch.save(model.quantum_head.state_dict(), checkpoint / "quantum_head.pt")\n    torch.save(\n        {\n            "epoch": epoch,\n            "optimizer": optimizer.state_dict(),\n            "scheduler": scheduler.state_dict(),\n        },\n        checkpoint / "training_state.pt",\n    )\n    config = {\n        "scorer": model.scorer,\n        "temperature": model.temperature,\n        "quantum_n_qubits": args.quantum_n_qubits,\n        "quantum_dropout": args.quantum_dropout,\n        "base_model": args.retriever_model,\n    }\n    (checkpoint / "experiment_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")\n\n\ndef train_retriever(args, device) -> None:\n    cache_dir = Path(args.cache_dir)\n    train_rows = read_jsonl(cache_dir / "train.jsonl")\n    valid_rows = read_jsonl(cache_dir / "valid.jsonl")\n    tokenizer = AutoTokenizer.from_pretrained(args.retriever_model)\n    train_dataset = FrozenRewardDataset(train_rows, tokenizer, args.query_length, args.candidate_length)\n    valid_dataset = FrozenRewardDataset(valid_rows, tokenizer, args.query_length, args.candidate_length)\n    generator = torch.Generator().manual_seed(args.seed)\n    train_loader = DataLoader(\n        train_dataset,\n        batch_size=args.train_batch_size,\n        shuffle=True,\n        generator=generator,\n        num_workers=args.num_workers,\n        pin_memory=torch.cuda.is_available(),\n    )\n    valid_loader = DataLoader(\n        valid_dataset,\n        batch_size=args.eval_batch_size,\n        shuffle=False,\n        num_workers=args.num_workers,\n        pin_memory=torch.cuda.is_available(),\n    )\n    model = ControlledRetriever(\n        args.retriever_model,\n        args.scorer,\n        args.quantum_n_qubits,\n        args.quantum_dropout,\n        args.temperature,\n    ).to(device)\n    optimizer = AdamW(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)\n    total_steps = max(len(train_loader) * args.epochs, 1)\n    scheduler = get_linear_schedule_with_warmup(\n        optimizer,\n        num_warmup_steps=int(total_steps * args.warmup_ratio),\n        num_training_steps=total_steps,\n    )\n    use_amp = args.fp16 and device.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)\n    output_dir = Path(args.output_dir) / args.scorer\n    output_dir.mkdir(parents=True, exist_ok=True)\n    metrics_path = output_dir / "metrics.jsonl"\n\n    initial = evaluate(model, valid_loader, tokenizer.pad_token_id, device)\n    print("Initial validation:", json.dumps(initial, indent=2))\n    with metrics_path.open("w", encoding="utf-8") as log:\n        log.write(json.dumps({"epoch": 0, "split": "valid", **initial}) + "\\n")\n\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        loss_sum = 0.0\n        seen = 0\n        started = time.time()\n        progress = tqdm(train_loader, desc=f"{args.scorer} epoch {epoch}/{args.epochs}")\n        for batch in progress:\n            query_ids = batch["query_ids"].to(device, non_blocking=True)\n            candidate_ids = batch["candidate_ids"].to(device, non_blocking=True)\n            labels = batch["label"].to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):\n                logits = model(query_ids, candidate_ids, tokenizer.pad_token_id)\n                loss = F.cross_entropy(logits.float(), labels)\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            loss_sum += loss.item() * labels.numel()\n            seen += labels.numel()\n            progress.set_postfix(loss=f"{loss_sum / max(seen, 1):.4f}")\n\n        train_record = {\n            "epoch": epoch,\n            "split": "train",\n            "loss": loss_sum / max(seen, 1),\n            "seconds": time.time() - started,\n        }\n        valid_record = {"epoch": epoch, "split": "valid", **evaluate(model, valid_loader, tokenizer.pad_token_id, device)}\n        print(json.dumps(train_record, indent=2))\n        print(json.dumps(valid_record, indent=2))\n        with metrics_path.open("a", encoding="utf-8") as log:\n            log.write(json.dumps(train_record) + "\\n")\n            log.write(json.dumps(valid_record) + "\\n")\n        save_checkpoint(model, tokenizer, output_dir, epoch, args, optimizer, scheduler)\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)\n    parser.add_argument("--stage", choices=("prepare", "train", "all"), default="all")\n    parser.add_argument("--scorer", choices=("cosine", "quantum_fidelity"), default="cosine")\n    parser.add_argument("--data-root", default="/kaggle/input/data4rlcoder")\n    parser.add_argument("--cache-dir", default="/kaggle/working/rlcoder_controlled/cache")\n    parser.add_argument("--output-dir", default="/kaggle/working/rlcoder_controlled/runs")\n    parser.add_argument("--retriever-model", default="microsoft/unixcoder-base")\n    parser.add_argument("--generator-model", default="deepseek-ai/deepseek-coder-1.3b-base")\n    parser.add_argument("--seed", type=int, default=123)\n    parser.add_argument("--train-examples", type=int, default=500)\n    parser.add_argument("--valid-examples", type=int, default=100)\n    parser.add_argument("--candidate-count", type=int, default=6)\n    parser.add_argument("--bm25-topk", type=int, default=50)\n    parser.add_argument("--block-lines", type=int, default=12)\n    parser.add_argument("--query-length", type=int, default=256)\n    parser.add_argument("--candidate-length", type=int, default=512)\n    parser.add_argument("--retriever-batch-size", type=int, default=32)\n    parser.add_argument("--generator-batch-size", type=int, default=2)\n    parser.add_argument("--generator-context-length", type=int, default=512)\n    parser.add_argument("--generator-crossfile-length", type=int, default=384)\n    parser.add_argument("--target-length", type=int, default=64)\n    parser.add_argument("--epochs", type=int, default=5)\n    parser.add_argument("--train-batch-size", type=int, default=4)\n    parser.add_argument("--eval-batch-size", type=int, default=8)\n    parser.add_argument("--learning-rate", type=float, default=5e-5)\n    parser.add_argument("--weight-decay", type=float, default=0.01)\n    parser.add_argument("--warmup-ratio", type=float, default=0.1)\n    parser.add_argument("--max-grad-norm", type=float, default=1.0)\n    parser.add_argument("--temperature", type=float, default=20.0)\n    parser.add_argument("--quantum-n-qubits", type=int, default=64)\n    parser.add_argument("--quantum-dropout", type=float, default=0.1)\n    parser.add_argument("--num-workers", type=int, default=2)\n    parser.add_argument("--fp16", action=argparse.BooleanOptionalAction, default=True)\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    seed_everything(args.seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    print(f"Device: {device}; stage={args.stage}; scorer={args.scorer}")\n    if args.stage in ("prepare", "all"):\n        prepare_reward_cache(args, device)\n    if args.stage in ("train", "all"):\n        train_retriever(args, device)\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
(ENGINE / 'rlcoder_compat.py').write_text('"""Extracted from local RLCoder source; regenerate with build_rlcoder_compat.py.\n\nCodeBlock, Example, sampling, natural block splitting and generator prompt/dataset\nare upstream implementations. Added bounded sampling attempts only. Identifier\npredicate originates in RLCoder/utils/eval_utils.py (Apache-2.0, Amazon.com).\nLanguage keyword handling follows Microsoft\'s MIT-licensed keywordlist.py.\n"""\nimport random\nimport re\nimport keyword\nimport torch\nimport pandas as pd\nfrom torch.utils.data import Dataset\nIDENTIFIER_REGEX = re.compile(\'[_a-zA-Z][_a-zA-Z0-9]*\')\nSOURCE_SHA256 = {\'datasets.py\': \'14348d392f60e71dc48fde2423abdf200baa6f5065af5d998d169a92278f20e7\', \'bm25.py\': \'dd3ecb482a1f3dafd4fcc98f606a287d8e891c7a189c27e7d566c8a750e3cc4e\', \'generator.py\': \'83911d18ac851b126dfef0ed9d7cb514ede028e07b08063d9ff342c5ef45f1a5\', \'utils/eval_utils.py\': \'ba5b78e79bac166b0ee6f4d7d458ff5516afc0acd5a33b0da345f95c87eb0f1a\', \'utils/keywords/java.txt\': \'807f36a9985ec3e75052b21ec3a8933f883deae84aa9cdac6c4779005e08752f\'}\nJAVA_KEYWORDS = frozenset([\'abstract\', \'assert\', \'boolean\', \'break\', \'byte\', \'case\', \'catch\', \'char\', \'class\', \'continue\', \'default\', \'do\', \'double\', \'else\', \'enum\', \'extends\', \'final\', \'finally\', \'float\', \'for\', \'if\', \'implements\', \'import\', \'instanceof\', \'int\', \'interface\', \'long\', \'native\', \'new\', \'package\', \'private\', \'protected\', \'public\', \'return\', \'short\', \'static\', \'strictfp\', \'super\', \'switch\', \'synchronized\', \'this\', \'throw\', \'throws\', \'transient\', \'try\', \'void\', \'volatile\', \'while\', \'var\', \'const\', \'goto\'])\n\ndef get_language_keywords(lang):\n    if lang == \'python\':\n        return frozenset(k for k in keyword.kwlist if k not in (\'True\', \'False\'))\n    if lang == \'java\':\n        return JAVA_KEYWORDS\n    raise ValueError(lang)\n\nclass CodeBlock(object):\n    def __init__(self, file_path, description, code_content, language, _type):\n        """\n        Represents a block of code.\n        :param file_path: The path to the code file.\n        :param description: The description of the code block.\n        :param code_content: The content of the code block.\n        """\n        self.file_path = file_path\n        self.code_content = code_content\n        self.description = description\n        self.language = language\n        self._type = _type\n\n    def __str__(self):\n        if self.language == "python":\n            comment_label = "#"\n        else:\n            comment_label = "//"\n        crossfile_context = "\\n".join([f"{comment_label} {cl}" for cl in  self.description.strip().split(\'\\n\') if cl]) + "\\n"\n        crossfile_context += "\\n".join([f"{comment_label} {cl}" for cl in  self.code_content.split(\'\\n\') if cl])\n        return crossfile_context.strip()\n\nclass Example(object):\n    def __init__(self, task_id, file_path, left_context, right_context, related_files, target_code, language):\n        """\n        Represents an example used for constructing a dataset.\n        :param task_id: Task ID.\n        :param file_path: File path.\n        :param left_context: The context to the left of the target code.\n        :param right_context: The context to the right of the target code.\n        :param related_files: A list of related files, each containing a path and text.\n        :param target_code: The target code snippet.\n        """\n        self.task_id = task_id\n        self.file_path = file_path\n        self.left_context = left_context\n        self.right_context = right_context\n        self.related_files = related_files\n        self.target_code = target_code\n        self.language = language\n\n    def __str__(self):\n        return (f"[Example]:\\n"\n                f"[Task ID]:\\n{self.task_id}\\n"\n                f"[Path]:\\n{self.file_path}\\n"\n                f"[Left Context]:\\n{self.left_context}\\n"\n                f"[Target Code]:\\n{self.target_code}\\n"\n                f"[Right Context]:\\n{self.right_context}\\n"\n                f"[Related Files]:\\n{len(self.related_files)} files\\n"\n        )\n\ndef load_train_and_valid_dataset(data_root):\n    """\n    Loads the training dataset.\n    :return: The training dataset.\n    """\n    training_datasets = []\n    validation_datasets = []\n    for language in ["python", "java"]:\n        data_frame = pd.read_parquet(f"{data_root}/github_repos/{language}/train.parquet")\n        all_data = []\n        temp_data = []\n        for x in data_frame[["path", "content", "first"]].values:\n            if x[-1]:  # At the start of a new file\n                if len(temp_data) > 1:\n                    all_data.append((temp_data,language))\n                temp_data = []\n            temp_data.append([x[0], x[1]])\n        training_datasets.extend(all_data[:2000])\n        validation_datasets.extend(all_data[2000:2200])\n    random.shuffle(training_datasets)\n    random.shuffle(validation_datasets)\n\n    return training_datasets, validation_datasets\n\ndef construct_dataset(raw_data, num_samples):\n    """\n    Builds a dataset.\n    :param raw_data: Raw data.\n    :param num_samples: The number of samples to generate.\n    :return: The list of constructed samples.\n    """\n    examples = []\n    data_index = 0\n    while len(examples) < num_samples:\n        example,language = raw_data[data_index % len(raw_data)]\n        data_index += 1\n        if data_index > max(1000, num_samples * 100):\n            raise RuntimeError("Insufficient eligible samples within bounded attempt budget")\n        selected_file = random.choice(example[1:])\n        related_files = [CodeBlock(x[0], f"file path: {x[0]}\\nlines: {0}-{len(x[1].splitlines())}", x[1], language, \'\') for x in example if x[0] != selected_file[0]]\n        path = selected_file[0]\n        selected_file_content = selected_file[1].split(" ")\n        try_count = 0\n\n        while try_count < 10:\n            end_line_number = int(len(selected_file_content) * random.uniform(0.2, 0.8))\n            left_context = " ".join(selected_file_content[:end_line_number])\n            target_length = random.randint(32, 64)\n            target = " ".join(selected_file_content[end_line_number:end_line_number + target_length])\n            right_context = " ".join(selected_file_content[end_line_number + target_length:])\n            if len(left_context.split()) > 80 and len(target.split()) > 8:\n                examples.append(\n                    Example(len(examples), path, left_context, right_context, related_files, target,language)\n                )\n                break\n            # if language == \'python\':\n            #     if len(left_context.split()) > 64 and len(target.split()) > 5:\n            #         examples.append(\n            #             Example(len(examples), path, left_context, right_context, related_files, target,language)\n            #         )\n            #         break\n            # elif language == \'java\':\n            #     if len(left_context.split()) > 80 and len(target.split()) > 8:\n            #         examples.append(\n            #             Example(len(examples), path, left_context, right_context, related_files, target,language)\n            #         )\n            #         break\n            try_count += 1\n    \n    return examples\n\ndef split_into_smaller_blocks(code_block, enable_fixed_block):\n    """\n    Split large blocks of code into smaller ones, each containing no more than 12 non-empty lines.\n    """\n    smaller_blocks = []\n\n    if enable_fixed_block:\n        lines = [line for line in code_block.code_content.split(\'\\n\') if line.strip() != \'\']\n        for i in range(0, min(len(lines),5000), 12):\n            start_line_offset = i\n            end_line_offset = min(i + 12, len(lines))\n            block_content = \'\\n\'.join(lines[start_line_offset:end_line_offset])\n            smaller_blocks.append(CodeBlock(code_block.file_path, \n                                            f"file path: {code_block.file_path}\\nlines: {start_line_offset}-{end_line_offset - 1}",\n                                            block_content,\n                                            code_block.language,\n                                            \'fixed_block\'))\n\n    else:\n        # Split the code by spaces, then reassemble it into blocks.\n        mini_blocks = []\n        current_block = [] \n        for line in code_block.code_content.splitlines(): \n            if line.strip() == \'\':  \n                if current_block: \n                    mini_blocks.append(current_block)\n                    current_block = []\n            else:\n                current_block.append(line)\n        if current_block: \n            mini_blocks.append(current_block)\n\n        max_len = 15\n        temp_mini_blocks = []\n        for mini_block in mini_blocks:\n            if len(mini_block) > max_len:\n                for idx in range(0, len(mini_block), max_len):\n                    temp_mini_blocks.append(mini_block[idx: idx+max_len])\n            else:\n                temp_mini_blocks.append(mini_block)\n        mini_blocks = temp_mini_blocks\n\n        current_content = []\n        total_lines = 0  \n        for block in mini_blocks:\n            if total_lines >= 5000:  \n                break  \n            if len(current_content) + len(block) <= 15:  \n                current_content.extend(block)\n                total_lines += len(block)  \n            else:  \n                if current_content:  \n                    smaller_blocks.append(CodeBlock(code_block.file_path, \n                                                    f"file path: {code_block.file_path}\\nlines: {total_lines-len(current_content)+1}-{total_lines}",\n                                                    \'\\n\'.join(current_content),\n                                                    code_block.language,\n                                                    \'mini_block\'))\n                current_content = block  \n                total_lines += len(block)  \n        if current_content:  \n            smaller_blocks.append(CodeBlock(code_block.file_path, \n                                            f"file path: {code_block.file_path}\\nlines: {total_lines-len(current_content)+1}-{total_lines}",\n                                            \'\\n\'.join(current_content),\n                                            code_block.language,\n                                            \'mini_block\'))\n        \n    return smaller_blocks\n\nclass GeneratorDataset(Dataset):\n    """\n    A dataset class for code generation.\n\n    Args:\n        args: Configuration parameters.\n        tokenizer: Tokenizer.\n        examples: A collection of examples.\n        retrieved_codeblocks: Retrieved code blocks.\n    """\n    def __init__(self, args, tokenizer, examples, retrieved_codeblocks, generation=False):\n        self.args = args\n        self.tokenizer = tokenizer\n        self.examples = examples\n        self.retrieved_codeblocks = retrieved_codeblocks\n        self.generation = generation\n\n    def __len__(self):\n        return len(self.examples)\n\n    def construct_prompts(self,example,retrieved_codeblocks):\n        filter_codeblocks = []\n        for x in retrieved_codeblocks:\n            # Keep only the blocks before the empty block\n            if x.file_path != "":\n                filter_codeblocks.append(x)\n            else:\n                break\n        crossfile_context = "\\n\\n".join([str(retrieved_codeblock) for retrieved_codeblock in filter_codeblocks])\n        crossfile_context = self.tokenizer.encode(crossfile_context[:self.args.generator_max_crossfile_length*10], add_special_tokens=False)[:self.args.generator_max_crossfile_length]\n        path_context = f"\\n\\n# file path: {example.file_path}\\n\\n"\n        path_context = self.tokenizer.encode(path_context, add_special_tokens=False)\n        allowed_prompt_length = self.args.generator_max_context_length - (len(crossfile_context)+len(path_context)+10)\n        infile_context = self.tokenizer.encode(example.left_context, add_special_tokens=False)[-allowed_prompt_length:]\n\n        prompt = self.tokenizer.decode(crossfile_context + path_context + infile_context)\n        return prompt\n\n    def __getitem__(self, idx):\n        example = self.examples[idx]\n        retrieved_codeblocks = self.retrieved_codeblocks[idx]\n        prompt = self.construct_prompts(example,retrieved_codeblocks)\n        \n        prompt_ids = self.tokenizer.encode(prompt)[-self.args.generator_max_context_length:]\n        if self.generation:\n             padding_length = self.args.generator_max_context_length - len(prompt_ids)\n             input_ids = [self.tokenizer.pad_token_id] * padding_length + prompt_ids\n             return torch.tensor(input_ids)\n\n        target_ids = self.tokenizer.encode(example.target_code, add_special_tokens=False)[:self.args.generator_max_generation_length]\n\n        input_ids = prompt_ids + target_ids\n        labels = [-100 for _ in prompt_ids] + target_ids\n\n        padding_length = self.args.generator_max_context_length + self.args.generator_max_generation_length - len(input_ids)\n        input_ids = [self.tokenizer.pad_token_id] * padding_length + input_ids\n        labels = [-100] * padding_length + labels \n\n        return torch.tensor(input_ids), torch.tensor(labels)\n\ndef is_identifier(token, lang=None):\n    return True if IDENTIFIER_REGEX.match(token) \\\n                   and (lang is None or token not in get_language_keywords(lang)) \\\n        else False\n', encoding="utf-8")
(ENGINE / 'online_learned_phase.py').write_text('"""Kaggle online RLCoder adaptation; see ONLINE_PLAN_VI.md.\n\nReuses data types/tokenization from controlled_comparison, not its fixed-pool loop.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport shutil\nimport subprocess\nimport gc\nimport json\nimport math\nimport os\nimport random\nimport time\nimport sys\nimport traceback\nimport numpy as np\nfrom dataclasses import asdict\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup\nfrom rank_bm25 import BM25Okapi\nimport rlcoder_compat as upstream\nfrom controlled_comparison import (\n    Candidate, CompletionExample, ControlledRetriever, FrozenRewardDataset,\n    NULL_CONTEXT, resolve_data_root,\n    rlcoder_tokenize, seed_everything, write_jsonl,\n)\n\nPIPELINE_VERSION = \'rlcoder-learned-phase-validation-v1\'\n\n\nclass OnlineRetriever(ControlledRetriever):\n    def forward(self, query_ids, candidate_ids, pad_token_id):\n        batch, count, length = candidate_ids.shape\n        query = self.embed(query_ids, pad_token_id)\n        docs = self.embed(candidate_ids.reshape(-1, length), pad_token_id).view(batch, count, -1)\n        # Trigonometry/complex overlaps must stay FP32 even under encoder AMP.\n        with torch.autocast(query.device.type, enabled=False):\n            if self.scorer == \'cosine\':\n                return F.cosine_similarity(query.float()[:, None], docs.float(), dim=-1) * self.temperature\n            if self.scorer == \'qiepsm_fidelity\':\n                return self.quantum_head.pairwise_fidelity(query.float(), docs.float()) * self.temperature\n            return self.quantum_head.pairwise_log_fidelity(query.float(), docs.float()) * self.temperature\n\n\ndef make_online_examples(groups, count, seed, split):\n    """Use upstream literal-space cut, retries and file selection unchanged."""\n    if not groups:\n        raise ValueError(f\'No repository groups for {split}; need the full training parquet\')\n    state = random.getstate()\n    try:\n        random.seed(seed)\n        examples = upstream.construct_dataset(groups, count)\n    finally:\n        random.setstate(state)\n    return [CompletionExample(f\'{split}-{i}\', ex.file_path, ex.left_context, ex.target_code,\n                ex.language, [(c.file_path,c.code_content) for c in ex.related_files])\n            for i, ex in enumerate(examples)]\n\n\ndef query_text(example):\n    return \'\\n\'.join([x for x in example.left_context.split(\'\\n\') if x.strip() != \'\'][-20:])\n\n\ndef codeblock(candidate):\n    return upstream.CodeBlock(candidate.file_path, candidate.description, candidate.code, candidate.language, \'\')\n\n\ndef candidate_text(candidate):\n    return str(codeblock(candidate))\n\n\ndef bm25_candidates(example, topk):\n    blocks = []\n    for path, content in example.related_files:\n        blocks.extend(upstream.split_into_smaller_blocks(\n            upstream.CodeBlock(path, \'\', content, example.language, \'\'), False))\n    if not blocks:\n        return []\n    index = BM25Okapi([block.code_content.lower().split() for block in blocks])\n    # Preserve upstream asymmetry: corpus lowercase; query not lowercased.\n    scores = index.get_scores(query_text(example).split())\n    order = sorted(range(len(scores)), key=lambda i:scores[i], reverse=True)[:topk]\n    return [Candidate(blocks[i].file_path, blocks[i].description, blocks[i].code_content,\n                      example.language) for i in order]\n\n\n@torch.no_grad()\ndef candidate_scores(model, tokenizer, example, candidates, args, device):\n    model.eval()\n    qids = torch.tensor([rlcoder_tokenize(query_text(example), tokenizer, args.query_length, True)], device=device)\n    q = model.embed(qids, tokenizer.pad_token_id).float()\n    scores = []\n    for start in range(0, len(candidates), args.encode_batch_size):\n        ids = torch.tensor([rlcoder_tokenize(candidate_text(c), tokenizer, args.candidate_length, False)\n                            for c in candidates[start:start + args.encode_batch_size]], device=device)\n        d = model.embed(ids, tokenizer.pad_token_id).float()[None]\n        if model.scorer == \'cosine\':\n            s = F.cosine_similarity(q[:, None], d, dim=-1)\n        elif model.scorer == \'qiepsm_fidelity\':\n            s = model.quantum_head.pairwise_fidelity(q, d)\n        else:\n            s = model.quantum_head.pairwise_log_fidelity(q, d)\n        scores.extend((s[0] * model.temperature).cpu().tolist())\n    return torch.tensor(scores)\n\n\ndef mine_pools(model, tokenizer, examples, args, device):\n    pools = []\n    for ex in examples:\n        if time.monotonic() >= getattr(args, \'_deadline\', float(\'inf\')):\n            raise TimeoutError(\'Time budget reached during mining; resume from latest.pt\')\n        lexical = bm25_candidates(ex, args.bm25_topk)\n        if not lexical:\n            raise RuntimeError(f\'No cross-file blocks: {ex.task_id}\')\n        scores = candidate_scores(model, tokenizer, ex, lexical, args, device)\n        selected = [lexical[i] for i in scores.numpy().argsort()[-(args.candidate_count-1):][::-1]]\n        if len(selected) < args.candidate_count-1:\n            selected += [Candidate(\'\', "Don\'t need cross file context to completion", \'\', ex.language, True)] * (args.candidate_count-1-len(selected))\n        selected.append(Candidate(\'\', NULL_CONTEXT, \'\', ex.language, True))\n        pools.append((ex, selected))\n    return pools\n\n\ndef reward_weights(labels, tokenizer, language, weighted=True):\n    """Literal upstream flattened-token weighting, including first-position quirk."""\n    if not weighted:\n        return torch.ones_like(labels, dtype=torch.float32)\n    tokens = [tokenizer.convert_ids_to_tokens(int(i)) if i != -100 else \'<pad>\'\n              for i in labels[:, 1:].reshape(-1)]\n    values = [5. if i < 1 else 3. if upstream.is_identifier(token, language)\n              and any(c.isalpha() or c.isdigit() or c == \'_\' for c in token) else 1.\n              for i, token in enumerate(tokens)]\n    weights = torch.ones_like(labels, dtype=torch.float32)\n    weights[:, 1:] = torch.tensor(values, device=labels.device).view(labels.shape[0], -1)\n    return weights\n\n\ndef generator_dataset(example, bundles, tokenizer, args):\n    cfg = SimpleNamespace(generator_max_context_length=args.generator_context_length,\n                          generator_max_crossfile_length=args.crossfile_length,\n                          generator_max_generation_length=args.target_length)\n    return upstream.GeneratorDataset(cfg, tokenizer, [example]*len(bundles),\n                                      [[codeblock(c) for c in bundle] for bundle in bundles])\n\n\ndef weighted_target_nll(logits, labels, weights):\n    """Only target tokens count; denominator matches RLCoder token-count convention."""\n    shifted = labels[:, 1:]\n    mask = shifted.ne(-100)\n    # Gather only supervised positions before FP32 conversion: saves vocabulary memory.\n    ce = F.cross_entropy(logits[:, :-1][mask].float(), shifted[mask], reduction=\'none\')\n    weighted = torch.zeros_like(shifted, dtype=torch.float32)\n    weighted[mask] = ce * weights[:, 1:][mask]\n    return weighted.sum(-1) / mask.sum(-1).clamp_min(1)\n\n\n@torch.inference_mode()\ndef score_pools(pools, args, device, combined=False):\n    tok = AutoTokenizer.from_pretrained(args.generator_model, use_fast=True)\n    tok.model_max_length = 1e10\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    gen = AutoModelForCausalLM.from_pretrained(args.generator_model,\n              torch_dtype=torch.float16 if device.type == \'cuda\' else torch.float32).to(device)\n    gen.eval().requires_grad_(False)\n    rows = []\n    for ex, candidates in pools:\n        if time.monotonic() >= getattr(args, \'_deadline\', float(\'inf\')):\n            raise TimeoutError(\'Time budget reached during reward scoring; resume from latest.pt\')\n        bundles = [candidates] if combined else [[c] for c in candidates]\n        sequences = generator_dataset(ex, bundles, tok, args)\n        losses = []\n        for start in range(0, len(sequences), args.generator_batch_size):\n            batch = [sequences[i] for i in range(start, min(start+args.generator_batch_size,len(sequences)))]\n            ids = torch.stack([x for x, _ in batch]).to(device)\n            labels = torch.stack([y for _, y in batch]).to(device)\n            # Upstream uses the first example\'s language throughout evaluate().\n            weights = reward_weights(labels, tok, pools[0][0].language, args.reward == \'weighted\')\n            mask = ids.ne(tok.pad_token_id)\n            output = gen(input_ids=ids, attention_mask=mask, use_cache=False)\n            losses.extend(weighted_target_nll(output.logits, labels, weights).cpu().tolist())\n            del output\n        if not all(math.isfinite(x) for x in losses):\n            raise RuntimeError(f\'Nonfinite generator reward: {ex.task_id}\')\n        rows.append({\'task_id\': ex.task_id, \'query\': query_text(ex), \'target\': ex.target_code,\n                     \'left_context\': ex.left_context, \'file_path\': ex.file_path,\n                     \'candidates\': [candidate_text(c) for c in candidates],\n                     \'candidate_meta\': [asdict(c) for c in candidates],\n                     \'generator_losses\': losses, \'label\': int(torch.tensor(losses).argmin())})\n    del gen\n    gc.collect()\n    if device.type == \'cuda\':\n        torch.cuda.empty_cache()\n    return rows\n\n\ndef check_resume_pipeline(checkpoint):\n    if checkpoint[\'config\'].get(\'pipeline_version\') != PIPELINE_VERSION:\n        raise ValueError(\'Checkpoint uses the older adaptation. Start a new v2 run; old candidate/reward caches are incompatible.\')\n\n\ndef retriever_loss(logits, costs, mode=\'hard\', teacher_temperature=.5, hard_weight=.5):\n    costs = costs.detach().float()\n    logp = F.log_softmax(logits.float(), dim=-1)\n    hard = F.nll_loss(logp, costs.argmin(-1))\n    if mode == \'hard\':\n        return hard\n    teacher = F.softmax(-(costs - costs.min(-1, keepdim=True).values) / teacher_temperature, dim=-1)\n    soft = -(teacher * logp).sum(-1).mean()\n    return soft if mode == \'soft\' else hard_weight * hard + (1-hard_weight) * soft\n\n\ndef collate_single(batch):\n    # One query and its candidate list per microbatch.\n    return {k: v.unsqueeze(0) for k, v in batch[0].items()}\n\n\ndef atomic_save(payload, path):\n    """Keep the previous complete file if writing is interrupted."""\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    try:\n        with temporary.open(\'wb\') as handle:\n            torch.save(payload, handle)\n            handle.flush()\n            os.fsync(handle.fileno())\n        os.replace(temporary, path)\n    finally:\n        # Failed writes must not leave a multi-GB .tmp consuming the remaining disk.\n        if temporary.exists():\n            temporary.unlink()\n\n\ndef capture_rng():\n    return {\'python\': random.getstate(), \'numpy\': np.random.get_state(),\n            \'torch\': torch.get_rng_state(),\n            \'cuda\': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}\n\n\ndef restore_rng(state):\n    random.setstate(state[\'python\'])\n    np.random.set_state(state[\'numpy\'])\n    torch.set_rng_state(state[\'torch\'].cpu())\n    if state[\'cuda\'] and torch.cuda.is_available():\n        torch.cuda.set_rng_state_all([x.cpu() for x in state[\'cuda\']])\n\n\ndef epoch_order(size, seed, epoch):\n    return torch.randperm(size, generator=torch.Generator().manual_seed(seed + epoch)).tolist()\n\n\ndef optimizer_updates_per_epoch(samples, train_batch_size, accumulation_steps):\n    microbatches = math.ceil(samples / train_batch_size)\n    return math.ceil(microbatches / accumulation_steps)\n\n\ndef accumulation_window_samples(remaining_samples, local_step, train_batch_size, accumulation_steps):\n    """Number of examples represented by the current accumulation window."""\n    window = local_step // accumulation_steps\n    start = window * accumulation_steps * train_batch_size\n    end = min(start + accumulation_steps * train_batch_size, remaining_samples)\n    return end - start\n\n\n@torch.no_grad()\ndef evaluate_rows(model, tokenizer, rows, args, device):\n    model.eval()\n    metrics = {\'selected_cost\': 0., \'stop_cost\': 0., \'regret\': 0., \'top1\': 0., \'stop_rate\': 0.}\n    for row in rows:\n        ex = CompletionExample(row[\'task_id\'], row[\'file_path\'], row[\'left_context\'], row[\'target\'], \'\', [])\n        candidates = [Candidate(**c) for c in row[\'candidate_meta\']]\n        index = int(candidate_scores(model, tokenizer, ex, candidates, args, device).argmax())\n        costs = row[\'generator_losses\']\n        metrics[\'selected_cost\'] += costs[index]\n        metrics[\'stop_cost\'] += costs[-1]\n        metrics[\'regret\'] += costs[index] - min(costs)\n        metrics[\'top1\'] += int(index == row[\'label\'])\n        metrics[\'stop_rate\'] += int(index == len(costs)-1)\n    return {k: v / len(rows) for k, v in metrics.items()}\n\n\ndef evaluate_online(model, tokenizer, examples, args, device):\n    """Upstream evaluation retrieval: rank BM25+STOP, concatenate until STOP."""\n    pools = []\n    for ex in examples:\n        if time.monotonic() >= args._deadline:\n            raise TimeoutError(\'Time budget reached during validation; resume from latest.pt\')\n        candidates = bm25_candidates(ex, args.bm25_topk)\n        candidates.append(Candidate(\'\', NULL_CONTEXT, \'\', ex.language, True))\n        scores = candidate_scores(model, tokenizer, ex, candidates, args, device)\n        indices = scores.numpy().argsort()[-args.candidate_count:][::-1]\n        selected = [candidates[i] for i in indices]\n        pools.append((ex, selected))\n    model.cpu()\n    torch.cuda.empty_cache()\n    rows = score_pools(pools, args, device, combined=True)\n    model.to(device)\n    return rows\n\n\ndef should_evaluate(epoch, eval_epochs):\n    return epoch > 0 and epoch in eval_epochs\n\n\ndef fidelity_diagnostics(fidelity, log_fidelity):\n    def number(value):\n        x = float(value)\n        return x if math.isfinite(x) else str(x)\n    result = {\'diagnostic_scope\': \'last_microbatch\',\n              \'fidelity_zero_fraction\': float(fidelity.eq(0).float().mean())}\n    for name, values in [(\'fidelity\', fidelity), (\'log_fidelity\', log_fidelity)]:\n        for stat, value in [(\'min\', values.min()), (\'mean\', values.mean()), (\'max\', values.max())]:\n            result[f\'{name}_{stat}\'] = number(value)\n    return result\n\n\nVALIDATION_PROTOCOL = \'crosscodeeval-test-subset-v1\'\n\n\ndef build_validation(groups, count, seed):\n    # groups retained in signature for compatibility; targets come from benchmark.\n    import pandas as pd\n    from huggingface_hub import hf_hub_download\n    result = []\n    for language in (\'python\', \'java\'):\n        path = hf_hub_download(\'nov3630/Data4RLCoder\', f\'cceval/{language}/test.parquet\',\n            repo_type=\'dataset\', revision=\'cb9639f20f2374d75e5b0b8e8650f1b20802bf9f\')\n        frame = pd.read_parquet(path)\n        records = json.loads(frame.to_json(orient=\'records\'))\n        if len({str(x[\'task_id\']) for x in records}) != len(records):\n            raise ValueError(\'Duplicate CrossCodeEval task IDs\')\n        records.sort(key=lambda x:hashlib.sha256(f"{seed}:{language}:{x[\'task_id\']}".encode()).hexdigest())\n        if count > len(records):\n            raise ValueError(\'Requested subset exceeds benchmark size\')\n        for row in records[:count]:\n            row[\'task_id\'] = str(row[\'task_id\'])\n            row[\'language\'] = language\n            row[\'key\'] = language+\'/\'+row[\'task_id\']\n            result.append(row)\n    return result\n\n\ndef validation_score(results):\n    if {r[\'language\'] for r in results} != {\'python\',\'java\'} or any(r[\'n\'] <= 0 for r in results):\n        raise ValueError(\'Both validation languages require nonempty completed results\')\n    return (sum(r[\'em\'] for r in results)/len(results), sum(r[\'es\'] for r in results)/len(results))\n\n\ndef export_phase(model, destination):\n    if model.scorer == \'qiepsm_learned_phase\':\n        from safetensors.torch import save_file\n        save_file({k:v.detach().cpu().contiguous() for k,v in model.quantum_head.state_dict().items()},\n                  str(destination/\'phase_head.safetensors\'))\n\n\ndef prune_epoch_checkpoints(output):\n    """Remove only trainer-owned epoch exports inside this run after latest is durable."""\n    root = Path(output).resolve()\n    for path in root.glob(\'checkpoint-epoch-*\'):\n        suffix = path.name.removeprefix(\'checkpoint-epoch-\')\n        if suffix.isdigit() and path.is_dir() and not path.is_symlink():\n            if path.resolve().parent != root:\n                raise ValueError(\'Checkpoint cleanup escaped output directory\')\n            shutil.rmtree(path)\n\n\ndef remove_eval_encoder(output):\n    root = Path(output).resolve()\n    path = root/\'eval_encoder\'\n    if path.exists():\n        if path.is_symlink() or path.resolve().parent != root:\n            raise ValueError(\'Unsafe eval encoder cleanup target\')\n        shutil.rmtree(path)\n\n\ndef export_best(state, model, tokenizer, destination, metadata):\n    from safetensors.torch import save_file\n    destination.mkdir(parents=True, exist_ok=True)\n    model.encoder.config.save_pretrained(destination)\n    tokenizer.save_pretrained(destination)\n    save_file({k[len(\'encoder.\'):]:v.contiguous() for k,v in state.items() if k.startswith(\'encoder.\')},\n              str(destination/\'model.safetensors\'), metadata={\'format\':\'pt\'})\n    phase = {k[len(\'quantum_head.\'):]:v.contiguous() for k,v in state.items() if k.startswith(\'quantum_head.\')}\n    if phase:\n        save_file(phase, str(destination/\'phase_head.safetensors\'))\n    (destination/\'selection.json\').write_text(json.dumps(metadata,indent=2))\n\n\ndef parse_args():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--data-root\', required=True)\n    p.add_argument(\'--output-dir\', default=\'/kaggle/working/rlcoder_online\')\n    p.add_argument(\'--generator-model\', default=\'deepseek-ai/deepseek-coder-1.3b-base\')\n    p.add_argument(\'--retriever-model\', default=\'microsoft/unixcoder-base\')\n    p.add_argument(\'--scorer\', choices=[\'qiepsm_learned_phase\', \'qiepsm_log_fidelity\', \'qiepsm_fidelity\', \'quantum_fidelity\', \'cosine\'], default=\'qiepsm_learned_phase\')\n    p.add_argument(\'--loss\', choices=[\'hard\', \'soft\', \'hybrid\'], default=\'hard\')\n    p.add_argument(\'--reward\', choices=[\'weighted\', \'nll\'], default=\'weighted\')\n    p.set_defaults(pipeline_version=PIPELINE_VERSION)\n    p.add_argument(\'--log-every-updates\', type=int, default=50)\n    p.add_argument(\'--benchmark-limit\', type=int, default=100, help=\'CrossCodeEval examples per language; 0 = full benchmark\')\n    p.add_argument(\'--eval-epochs\', nargs=\'+\', type=int, default=list(range(2, 17, 2)))\n    p.add_argument(\'--resume\', help=\'Path to your own trusted latest.pt or epoch training.pt\')\n    p.add_argument(\'--save-every-updates\', type=int, default=10)\n    p.add_argument(\'--save-every-minutes\', type=float, default=10.)\n    p.add_argument(\'--max-runtime-minutes\', type=float, default=0., help=\'0 disables graceful time limit\')\n    for name, value in dict(samples_per_epoch=2000, epochs=16, valid_examples=200, candidate_count=6,\n                            bm25_topk=60, query_length=256, candidate_length=256,\n                            generator_context_length=768, crossfile_length=256, target_length=64,\n                            encode_batch_size=16, generator_batch_size=2, train_batch_size=4,\n                            accumulation_steps=4,\n                            n_qubits=32, seed=123).items():\n        p.add_argument(\'--\'+name.replace(\'_\',\'-\'), type=int, default=value)\n    for name, value in dict(encoder_lr=1e-5, head_lr=5e-5, score_scale=20.,\n                            teacher_temperature=.5, hard_weight=.5, warmup_ratio=.1).items():\n        p.add_argument(\'--\'+name.replace(\'_\',\'-\'), type=float, default=value)\n    args = p.parse_args()\n    if args.scorer in (\'qiepsm_log_fidelity\', \'qiepsm_learned_phase\') and not any(a.split(\'=\')[0] == \'--score-scale\' for a in sys.argv[1:]):\n        args.score_scale = 0.1\n    if any(epoch < 1 for epoch in args.eval_epochs):\n        p.error(\'eval-epochs must contain positive epochs\')\n    if args.benchmark_limit < 0:\n        p.error(\'benchmark-limit must be >= 0\')\n    for name in (\'samples_per_epoch\',\'epochs\',\'valid_examples\',\'bm25_topk\',\'encode_batch_size\',\n                 \'generator_batch_size\',\'train_batch_size\',\'accumulation_steps\',\'n_qubits\',\n                 \'log_every_updates\',\n                 \'generator_context_length\',\'crossfile_length\',\'target_length\'):\n        if getattr(args, name) < 1:\n            p.error(f\'{name} must be positive\')\n    if args.candidate_count < 2 or args.bm25_topk < args.candidate_count-1:\n        p.error(\'Need >=2 candidates and sufficient BM25 topk\')\n    if args.teacher_temperature <= 0 or not 0 <= args.hard_weight <= 1:\n        p.error(\'Invalid loss temperatures/weights\')\n    if args.save_every_updates < 1 or args.save_every_minutes <= 0 or args.max_runtime_minutes < 0:\n        p.error(\'Invalid checkpoint intervals/time budget\')\n    return args\n\n\ndef main():\n    args = parse_args()\n    runtime_started = time.monotonic()\n    resumed = None\n    if args.resume:\n        # Only load checkpoints produced by this script that you trust.\n        resumed = torch.load(args.resume, map_location=\'cpu\', weights_only=False)\n        check_resume_pipeline(resumed)\n        if args.scorer != resumed[\'config\'][\'scorer\']:\n            raise ValueError(\'Resume scorer differs from checkpoint; use a matching scorer or start a fresh run.\')\n        runtime_keys = {\'log_every_updates\', \'resume\', \'data_root\', \'output_dir\', \'save_every_updates\',\n                        \'save_every_minutes\', \'max_runtime_minutes\'}\n        for key, value in resumed[\'config\'].items():\n            if key not in runtime_keys:\n                setattr(args, key, value)\n        # Checkpoints produced before batched retriever training used microbatch 1.\n        if \'train_batch_size\' not in resumed[\'config\']:\n            args.train_batch_size = 1\n    if args.scorer in (\'qiepsm_fidelity\', \'qiepsm_log_fidelity\', \'qiepsm_learned_phase\'):\n        if (args.scorer == \'qiepsm_fidelity\' and args.score_scale != 20.) or args.loss != \'hard\':\n            raise ValueError(\'Faithful QIEPSM mode requires score-scale=20 and loss=hard.\')\n        # Metadata reflects the actual uncompressed state size, never the legacy 32.\n        from transformers import AutoConfig\n        args.n_qubits = AutoConfig.from_pretrained(args.retriever_model).hidden_size\n        args.fidelity_implementation = args.scorer + \'-uncompressed-v2\'\n        if args.score_scale <= 0:\n            raise ValueError(\'score-scale must be positive\')\n    seed_everything(args.seed)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    if device.type != \'cuda\':\n        raise RuntimeError(\'Enable a Kaggle GPU; local CPU is for unit tests only\')\n    out = Path(args.output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    phase_file = out/\'phase.json\'\n    def phase(name, **details):\n        payload = {\'phase\':name, \'time\':time.strftime(\'%Y-%m-%dT%H:%M:%S\'), **details}\n        temporary = phase_file.with_suffix(\'.json.tmp\')\n        temporary.write_text(json.dumps(payload, indent=2), encoding=\'utf-8\')\n        os.replace(temporary, phase_file)\n        print(\'PHASE:\', name, details, flush=True)\n    def failure_hook(exc_type, exc, tb):\n        report = \'\'.join(traceback.format_exception(exc_type, exc, tb))\n        try:\n            (out/\'failure.txt\').write_text(report, encoding=\'utf-8\')\n            phase(\'failed\', exception=exc_type.__name__, message=str(exc))\n        finally:\n            sys.__excepthook__(exc_type, exc, tb)\n    sys.excepthook = failure_hook\n    if (out/\'config.json\').exists() and not args.resume:\n        raise FileExistsError(\'Use a fresh output directory to avoid mixing runs\')\n    (out/\'config.json\').write_text(json.dumps(vars(args), indent=2), encoding=\'utf-8\')\n    import importlib.metadata\n    (out/\'environment.json\').write_text(json.dumps({\n        \'packages\': {p: importlib.metadata.version(p) for p in (\'torch\',\'transformers\',\'numpy\',\'pandas\',\'rank-bm25\')},\n        \'gpu\': torch.cuda.get_device_name(0), \'upstream_source_sha256\':upstream.SOURCE_SHA256}, indent=2))\n    phase(\'load_dataset\')\n    train_groups, valid_groups = upstream.load_train_and_valid_dataset(resolve_data_root(args.data_root))\n    phase(\'load_retriever\', model=args.retriever_model)\n    tok = AutoTokenizer.from_pretrained(args.retriever_model)\n    model = OnlineRetriever(args.retriever_model, args.scorer, args.n_qubits, 0., args.score_scale)\n    model.encoder.gradient_checkpointing_enable()\n    model.encoder.config.use_cache = False\n    model.to(device)\n    groups = [{\'params\': model.encoder.parameters(), \'lr\': args.encoder_lr}]\n    if args.scorer in (\'quantum_fidelity\', \'qiepsm_learned_phase\'):\n        groups.append({\'params\': model.quantum_head.parameters(), \'lr\': args.head_lr})\n    optimizer = torch.optim.AdamW(groups, weight_decay=.01)\n    total_steps = optimizer_updates_per_epoch(\n        args.samples_per_epoch, args.train_batch_size, args.accumulation_steps\n    ) * args.epochs\n    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps*args.warmup_ratio), total_steps)\n    scaler = torch.amp.GradScaler(\'cuda\')\n    progress = {\'epoch\':1, \'next_batch\':0, \'updates\':0, \'loss_sum\':0.}\n    valid_rows, rows = None, None\n    best = None\n    best_model = None\n    if resumed:\n        model.load_state_dict(resumed[\'model\'])\n        optimizer.load_state_dict(resumed[\'optimizer\'])\n        scheduler.load_state_dict(resumed[\'scheduler\'])\n        scaler.load_state_dict(resumed[\'scaler\'])\n        progress = resumed[\'progress\']\n        best, best_model = resumed.get(\'best\'), resumed.get(\'best_model\')\n        valid_rows, rows = resumed[\'valid_rows\'], resumed[\'train_rows\']\n        if resumed[\'config\'].get(\'validation_protocol\') != VALIDATION_PROTOCOL:\n            valid_rows, best, best_model = None, None, None\n            print(\'Validation changed to CrossCodeEval subset; old best reset. Training state retained.\', flush=True)\n        restore_rng(resumed[\'rng\'])\n        del resumed\n    if valid_rows is None:\n        valid_rows = build_validation(valid_groups, args.valid_examples, args.seed)\n    args.validation_protocol = VALIDATION_PROTOCOL\n    validation_path = out/\'validation_crosscodeeval.jsonl\'\n    write_jsonl(validation_path, valid_rows)\n    if best_model is not None:\n        export_best(best_model, model, tok, out/\'best_crosscodeeval\', best)\n    checkpoint_config = vars(args).copy()\n    args._deadline = (runtime_started + args.max_runtime_minutes*60\n                      if args.max_runtime_minutes else float(\'inf\'))\n    last_saved = time.monotonic()\n    def save(path=None):\n        nonlocal last_saved\n        atomic_save({\'format_version\':1, \'model\':model.state_dict(),\n                     \'optimizer\':optimizer.state_dict(), \'scheduler\':scheduler.state_dict(),\n                     \'scaler\':scaler.state_dict(), \'config\':checkpoint_config,\n                     \'progress\':progress, \'valid_rows\':valid_rows, \'train_rows\':rows,\n                     \'best\':best, \'best_model\':best_model, \'rng\':capture_rng()}, path or out/\'latest.pt\')\n        last_saved = time.monotonic()\n    # A self-contained recoverable state exists before any expensive generator scoring.\n    save()\n    # Migration of old outputs is safe only after a complete latest.pt exists.\n    prune_epoch_checkpoints(out)\n    remove_eval_encoder(out)\n    def log(record):\n        print(json.dumps(record), flush=True)\n        with (out/\'metrics.jsonl\').open(\'a\', encoding=\'utf-8\') as f:\n            f.write(json.dumps(record)+\'\\n\')\n    for epoch in range(progress[\'epoch\'], args.epochs+1):\n        started = time.time()\n        if time.monotonic() >= args._deadline:\n            print(\'Time budget reached. Resume from\', out/\'latest.pt\')\n            return\n        if rows is None:\n            phase(\'mine_train_candidates\', epoch=epoch, examples=args.samples_per_epoch)\n            examples = make_online_examples(train_groups, args.samples_per_epoch, args.seed+epoch, f\'train-e{epoch}\')\n            pools = mine_pools(model, tok, examples, args, device)\n            phase(\'score_train_rewards\', epoch=epoch, candidates=sum(len(x[1]) for x in pools),\n                  generator=args.generator_model)\n            model.cpu()\n            torch.cuda.empty_cache()\n            rows = score_pools(pools, args, device)\n            model.to(device)\n            save()\n        # Current reward rows already live in latest.pt; avoid duplicating code text per epoch.\n        model.to(device).train()\n        phase(\'train_retriever\', epoch=epoch, start_batch=progress[\'next_batch\'])\n        dataset = FrozenRewardDataset(rows, tok, args.query_length, args.candidate_length)\n        order = epoch_order(len(dataset), args.seed, epoch)[progress[\'next_batch\']:]\n        # Separate RNG prevents iterator construction from changing resumed dropout RNG.\n        loader = DataLoader(dataset, batch_size=args.train_batch_size, sampler=order,\n                            generator=torch.Generator().manual_seed(args.seed+epoch))\n        optimizer.zero_grad(set_to_none=True)\n        loss_sum = progress[\'loss_sum\']\n        remaining_samples = len(order)\n        for local_step, batch in enumerate(loader):\n            batch = {k:v.to(device) for k,v in batch.items()}\n            with torch.autocast(\'cuda\', dtype=torch.float16):\n                logits = model(batch[\'query_ids\'], batch[\'candidate_ids\'], tok.pad_token_id)\n            loss = retriever_loss(logits, batch[\'generator_losses\'], args.loss,\n                                  args.teacher_temperature, args.hard_weight)\n            if not torch.isfinite(loss):\n                raise RuntimeError(\'Nonfinite retriever loss\')\n            batch_samples = batch[\'label\'].shape[0]\n            window_samples = accumulation_window_samples(\n                remaining_samples, local_step, args.train_batch_size, args.accumulation_steps\n            )\n            scaler.scale(loss * batch_samples / window_samples).backward()\n            loss_sum += loss.item() * batch_samples\n            last_microbatch = local_step + 1 == len(loader)\n            if (local_step+1) % args.accumulation_steps == 0 or last_microbatch:\n                scaler.unscale_(optimizer)\n                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() >= old_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n                progress.update(next_batch=progress[\'next_batch\'] + window_samples,\n                                loss_sum=loss_sum, updates=progress[\'updates\']+1)\n                if progress[\'updates\'] % args.log_every_updates == 0 or last_microbatch:\n                    record = {\'event\': \'train_progress\', \'epoch\': epoch,\n                        \'updates\': progress[\'updates\'], \'samples\': progress[\'next_batch\'],\n                        \'train_loss\': loss_sum / progress[\'next_batch\'],\n                        \'gradient_norm\': float(grad_norm)}\n                    if args.scorer in (\'qiepsm_fidelity\', \'qiepsm_log_fidelity\', \'qiepsm_learned_phase\'):\n                        log_fidelity = model.quantum_head.last_log_fidelity\n                        fidelity = log_fidelity.exp()\n                        record.update(logit_min=float(logits.detach().min()), logit_max=float(logits.detach().max()), score_scale=args.score_scale)\n                        record.update(fidelity_diagnostics(fidelity, log_fidelity))\n                        with (out/\'qiepsm_score_diagnostics.jsonl\').open(\'a\', encoding=\'utf-8\') as handle:\n                            handle.write(json.dumps(record)+\'\\n\')\n                    log(record)\n                should_stop = time.monotonic() >= args._deadline\n                if (progress[\'updates\'] % args.save_every_updates == 0 or\n                    time.monotonic()-last_saved >= args.save_every_minutes*60 or should_stop):\n                    save()\n                if should_stop:\n                    print(\'Saved at optimizer boundary. Resume from\', out/\'latest.pt\')\n                    return\n        save()  # Protect the final update before validation/export.\n        if should_evaluate(epoch, args.eval_epochs):\n            eval_encoder = out/\'eval_encoder\'\n            eval_encoder.mkdir(exist_ok=True)\n            model.encoder.save_pretrained(eval_encoder)\n            tok.save_pretrained(eval_encoder)\n            export_phase(model, eval_encoder)\n            phase(\'generate_benchmark\', epoch=epoch)\n            if args.scorer not in (\'qiepsm_fidelity\', \'qiepsm_log_fidelity\', \'qiepsm_learned_phase\'):\n                raise ValueError(\'Integrated generation benchmark currently requires qiepsm_fidelity\')\n            result_dir = out/f\'crosscodeeval-subset-epoch-{epoch}\'\n            command = [sys.executable, \'-u\', str(Path(__file__).parent/\'generation_eval\'/\'benchmark.py\'),\n                \'--checkpoint-root\', str(eval_encoder), \'--output\', str(result_dir),\n                \'--methods\', args.scorer, \'--limit\', \'0\', \'--validation-jsonl\', str(validation_path),\n                \'--seed\', str(args.seed), \'--generator\', args.generator_model]\n            # Free retriever and optimizer GPU storage while the child loads generator.\n            model.cpu()\n            for state in optimizer.state.values():\n                for key, value in state.items():\n                    if torch.is_tensor(value):\n                        state[key] = value.cpu()\n            gc.collect()\n            torch.cuda.empty_cache()\n            remaining = args._deadline-time.monotonic()\n            try:\n                subprocess.run(command, check=True,\n                    timeout=max(1, remaining) if math.isfinite(remaining) else None)\n            except subprocess.TimeoutExpired as exc:\n                raise TimeoutError(\'Benchmark time budget reached; checkpoint saved. Resume to finish this epoch benchmark.\') from exc\n            model.to(device)\n            for state in optimizer.state.values():\n                for key, value in state.items():\n                    if torch.is_tensor(value) and key != \'step\':\n                        state[key] = value.to(device)\n            report = json.loads((result_dir/\'metrics.json\').read_text())\n            score = validation_score(report[\'results\'])\n            log({\'event\':\'validation\', \'epoch\':epoch, \'macro_em\':score[0], \'macro_es\':score[1]})\n            if best is None or score > (best[\'macro_em\'], best[\'macro_es\']):\n                best = {\'epoch\':epoch, \'macro_em\':score[0], \'macro_es\':score[1],\n                        \'selection\':\'validation macro EM; ES tie-break\'}\n                best_model = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n                export_best(best_model, model, tok, out/\'best_crosscodeeval\', best)\n                log({\'event\':\'new_best\', **best})\n        log({\'epoch\':epoch, \'train_loss\':loss_sum/len(dataset), \'samples\':len(rows),\n             \'seconds\':time.time()-started, \'peak_vram_gb\':torch.cuda.max_memory_allocated()/2**30})\n        progress.update(epoch=epoch+1, next_batch=0, loss_sum=0.)\n        rows = None\n        save()\n        remove_eval_encoder(out)\n    phase(\'complete\', epochs=args.epochs)\n    print(\'Finished. Save the Kaggle notebook version to retain /kaggle/working outputs.\')\n\n\nif __name__ == \'__main__\':\n    try:\n        main()\n    except TimeoutError as exc:\n        print(str(exc), flush=True)\n', encoding="utf-8")
p = ENGINE / "generation_eval" / 'benchmark.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_bytes(b'"""Evaluate trained RLCoder retrievers on CrossCodeEval code completion.\r\n\r\nThe retriever ranks the upstream BM25 shortlist (including the learned STOP\r\ncandidate).  A frozen DeepSeek-Coder generator then produces completions.  EM\r\nand Edit Similarity are the primary metrics.\r\n"""\r\nfrom __future__ import annotations\r\n\r\nimport argparse\r\nimport gc\r\nimport hashlib\r\nimport json\r\nimport math\r\nimport os\r\nimport time\r\nfrom dataclasses import dataclass\r\nfrom pathlib import Path\r\n\r\nimport numpy as np\r\nimport pandas as pd\r\nfrom rank_bm25 import BM25Okapi\r\n\r\nfrom rlcoder_compat import CodeBlock, split_into_smaller_blocks\r\nfrom vendor_aligncoder.utils import eval_utils as metrics\r\n\r\nDATASET = "nov3630/Data4RLCoder"\r\nDATASET_REVISION = "cb9639f20f2374d75e5b0b8e8650f1b20802bf9f"\r\nGENERATOR = "deepseek-ai/deepseek-coder-1.3b-base"\r\nGENERATOR_REVISION = "c919139c3a9b4070729c8b2cca4847ab29ca8d94"\r\nSCHEMA_VERSION = 1\r\nSTOP_TEXT = "Don\'t need cross file context for completion"\r\n\r\n\r\ndef digest(value) -> str:\r\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\r\n\r\n\r\ndef file_hash(path: Path) -> str:\r\n    h = hashlib.sha256()\r\n    with path.open("rb") as stream:\r\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\r\n            h.update(block)\r\n    return h.hexdigest()\r\n\r\n\r\ndef write_json(path: Path, value) -> None:\r\n    temp = path.with_suffix(path.suffix + ".tmp")\r\n    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\r\n    temp.replace(path)\r\n\r\n\r\ndef read_jsonl(path: Path) -> dict:\r\n    if not path.exists():\r\n        return {}\r\n    result = {}\r\n    with path.open(encoding="utf-8") as stream:\r\n        for line_number, line in enumerate(stream, 1):\r\n            try:\r\n                row = json.loads(line)\r\n            except json.JSONDecodeError as exc:\r\n                raise ValueError(f"Incomplete JSONL at {path}:{line_number}") from exc\r\n            if row["key"] in result:\r\n                raise ValueError(f"Duplicate key {row[\'key\']} in {path}")\r\n            result[row["key"]] = row\r\n    return result\r\n\r\n\r\ndef append_jsonl(path: Path, row: dict) -> None:\r\n    with path.open("a", encoding="utf-8") as stream:\r\n        stream.write(json.dumps(row, ensure_ascii=False) + "\\n")\r\n        stream.flush()\r\n        os.fsync(stream.fileno())\r\n\r\n\r\n@dataclass\r\nclass Example:\r\n    task_id: str\r\n    file_path: str\r\n    left_context: str\r\n    target: str\r\n    language: str\r\n    related_files: list[CodeBlock]\r\n\r\n\r\ndef make_example(row: dict) -> Example:\r\n    related = row["crossfile_context"]\r\n    if isinstance(related, str):\r\n        related = json.loads(related)\r\n    blocks = [CodeBlock(x["path"], f"file path: {x[\'path\']}\\nlines: 0-{len(x[\'text\'].splitlines())}",\r\n                        x["text"], row["language"], "") for x in related]\r\n    return Example(str(row["task_id"]), row["path"], row["left_context"], row["groundtruth"],\r\n                   row["language"], blocks)\r\n\r\n\r\ndef select_rows(frame: pd.DataFrame, language: str, limit: int, seed: int) -> list[dict]:\r\n    rows = frame.to_dict("records")\r\n    rows.sort(key=lambda r: hashlib.sha256(f"{seed}:{language}:{r[\'task_id\']}".encode()).hexdigest())\r\n    return rows if limit == 0 else rows[:limit]\r\n\r\n\r\ndef cceval_filename(language: str) -> str:\r\n    """Return the path used by the pinned Data4RLCoder revision.\r\n\r\n    Data4RLCoder stores its top-level datasets directly under ``cceval/``,\r\n    ``github_repos/`` and ``repoeval/``.  There is no enclosing ``data/``\r\n    directory in the Hugging Face repository.\r\n    """\r\n    return f"cceval/{language}/test.parquet"\r\n\r\n\r\ndef prepare_data(args) -> tuple[list[dict], dict]:\r\n    if getattr(args, \'validation_jsonl\', None):\r\n        path = Path(args.validation_jsonl)\r\n        rows = list(read_jsonl(path).values())\r\n        if not rows or {r[\'language\'] for r in rows} != set(args.languages):\r\n            raise ValueError(\'Validation language coverage mismatch\')\r\n        return rows, {\'validation\': {\'sha256\':file_hash(path), \'total\':len(rows)}}\r\n    from huggingface_hub import hf_hub_download\r\n    rows, sources = [], {}\r\n    for language in args.languages:\r\n        if args.data_root:\r\n            root = Path(args.data_root)\r\n            choices = [root / "cceval" / language / "test.parquet",\r\n                       root / "data" / "cceval" / language / "test.parquet"]\r\n            path = next((p for p in choices if p.exists()), None)\r\n            if path is None:\r\n                raise FileNotFoundError(f"Missing cceval/{language}/test.parquet under {root}")\r\n        else:\r\n            path = Path(hf_hub_download(DATASET, cceval_filename(language),\r\n                                        repo_type="dataset", revision=args.dataset_revision))\r\n        frame = pd.read_parquet(path)\r\n        if frame.task_id.astype(str).duplicated().any():\r\n            raise ValueError(f"Duplicate task IDs in {language}")\r\n        sources[language] = {"sha256": file_hash(path), "total": len(frame)}\r\n        for row in select_rows(frame, language, args.limit, args.seed):\r\n            row["task_id"] = str(row["task_id"])\r\n            row["language"] = language\r\n            row["key"] = f"{language}/{row[\'task_id\']}"\r\n            rows.append(row)\r\n    return rows, sources\r\n\r\n\r\ndef query_text(example: Example) -> str:\r\n    return "\\n".join([x for x in example.left_context.split("\\n") if x.strip()][-20:])\r\n\r\n\r\ndef bm25_pool(example: Example, topk: int) -> list[CodeBlock]:\r\n    blocks = []\r\n    for file_block in example.related_files:\r\n        blocks.extend(split_into_smaller_blocks(file_block, False))\r\n    if not blocks:\r\n        return []\r\n    corpus = [b.code_content.lower().split() for b in blocks]\r\n    scores = BM25Okapi(corpus).get_scores(query_text(example).split())\r\n    order = sorted(range(len(blocks)), key=lambda i: scores[i], reverse=True)[:topk]\r\n    return [blocks[i] for i in order]\r\n\r\n\r\ndef render_block(block: CodeBlock) -> str:\r\n    return str(block)\r\n\r\n\r\ndef block_dict(block: CodeBlock) -> dict:\r\n    return {"file_path": block.file_path, "description": block.description,\r\n            "code_content": block.code_content, "language": block.language, "type": block._type}\r\n\r\n\r\ndef dict_block(value: dict) -> CodeBlock:\r\n    return CodeBlock(value["file_path"], value["description"], value["code_content"],\r\n                     value["language"], value.get("type", ""))\r\n\r\n\r\ndef rlcoder_ids(text: str, tokenizer, max_length: int, query: bool):\r\n    tokens = tokenizer.tokenize(text)\r\n    room = max_length - 4\r\n    tokens = tokens[-room:] if query else tokens[:room]\r\n    tokens = [tokenizer.cls_token, "<encoder-only>", tokenizer.sep_token] + tokens + [tokenizer.sep_token]\r\n    ids = tokenizer.convert_tokens_to_ids(tokens)\r\n    return ids + [tokenizer.pad_token_id] * (max_length - len(ids))\r\n\r\n\r\nclass QuantumHead:\r\n    def __init__(self, hidden_size: int, n_qubits: int, path: Path, device: str):\r\n        import torch\r\n        from safetensors.torch import load_file\r\n        self.device, self.n_qubits = device, n_qubits\r\n        self.norm = torch.nn.LayerNorm(hidden_size).to(device)\r\n        self.linear = torch.nn.Linear(hidden_size, 2 * n_qubits).to(device)\r\n        state = load_file(str(path))\r\n        expected = (2 * n_qubits, hidden_size)\r\n        actual = tuple(state["projection.2.weight"].shape)\r\n        if actual != expected:\r\n            raise ValueError(\r\n                f"Quantum head shape {actual} does not match n_qubits={n_qubits}, "\r\n                f"hidden_size={hidden_size}; expected {expected}")\r\n        self.norm.load_state_dict({"weight": state["projection.0.weight"], "bias": state["projection.0.bias"]})\r\n        self.linear.load_state_dict({"weight": state["projection.2.weight"], "bias": state["projection.2.bias"]})\r\n        self.norm.eval(); self.linear.eval()\r\n\r\n    def score(self, query, docs):\r\n        import torch\r\n        with torch.inference_mode():\r\n            def states(x):\r\n                theta_raw, phi_raw = self.linear(self.norm(x.float())).chunk(2, -1)\r\n                theta, phi = torch.sigmoid(theta_raw) * math.pi, torch.tanh(phi_raw) * math.pi\r\n                a0 = torch.cos(theta / 2).to(torch.complex64)\r\n                a1 = torch.sin(theta / 2).to(torch.complex64) * torch.exp(1j * phi.float())\r\n                return torch.stack((a0, a1), -1)\r\n            q, d = states(query).unsqueeze(1), states(docs.unsqueeze(0))\r\n            fidelity = (q.conj() * d).sum(-1).abs().square()\r\n            return torch.log(fidelity.clamp_min(1e-7)).mean(-1)[0]\r\n\r\n\r\ndef load_local_encoder(checkpoint: Path, device: str, dtype):\r\n    """Load an extracted encoder without relying on Transformers metadata.\r\n\r\n    The epoch-8 encoder tensors were extracted from ``training.pt`` with\r\n    safetensors metadata describing their experiment source.  Recent\r\n    Transformers releases only accept ``metadata={"format": "pt"}`` in\r\n    ``from_pretrained``.  Loading the exact state dict into the pinned local\r\n    config is equivalent and works with both the older and newer releases.\r\n    """\r\n    from safetensors.torch import load_file\r\n    from transformers import AutoConfig, AutoModel\r\n\r\n    config = AutoConfig.from_pretrained(checkpoint, local_files_only=True)\r\n    # Do not pass ``dtype``/``torch_dtype`` here: support differs between the\r\n    # Transformers releases used locally and by Kaggle.  PyTorch casting is\r\n    # stable across both environments.\r\n    model = AutoModel.from_config(config).to(dtype=dtype)\r\n    state = load_file(str(checkpoint / "model.safetensors"), device="cpu")\r\n    incompatible = model.load_state_dict(state, strict=True)\r\n    if incompatible.missing_keys or incompatible.unexpected_keys:\r\n        raise ValueError(\r\n            f"Encoder state mismatch: missing={incompatible.missing_keys}, "\r\n            f"unexpected={incompatible.unexpected_keys}")\r\n    return model.to(device).eval()\r\n\r\n\r\nclass RLCoderRetriever:\r\n    def __init__(self, checkpoint: Path, scorer: str, args):\r\n        import torch\r\n        from transformers import AutoTokenizer\r\n        self.device, self.scorer, self.args = args.device, scorer, args\r\n        self.tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True)\r\n        dtype = torch.float16 if args.device.startswith("cuda") else torch.float32\r\n        self.model = load_local_encoder(checkpoint, args.device, dtype)\r\n        self.head = None\r\n        if scorer == \'qiepsm_learned_phase\':\r\n            from controlled_comparison import QIEPSMLearnedPhase\r\n            from safetensors.torch import load_file\r\n            self.head = QIEPSMLearnedPhase(self.model.config.hidden_size).to(args.device).eval()\r\n            self.head.load_state_dict(load_file(str(checkpoint/\'phase_head.safetensors\')), strict=True)\r\n        if scorer == "quantum":\r\n            head_path = checkpoint / "quantum_head.safetensors"\r\n            if not head_path.exists():\r\n                raise FileNotFoundError(f"Quantum head missing: {head_path}")\r\n            self.head = QuantumHead(self.model.config.hidden_size, args.n_qubits, head_path, args.device)\r\n\r\n    def encode(self, texts, query: bool):\r\n        import torch\r\n        output = []\r\n        length = self.args.query_tokens if query else self.args.candidate_tokens\r\n        with torch.inference_mode():\r\n            for start in range(0, len(texts), self.args.embedding_batch_size):\r\n                ids = torch.tensor([rlcoder_ids(x, self.tokenizer, length, query)\r\n                                    for x in texts[start:start + self.args.embedding_batch_size]],\r\n                                   device=self.device)\r\n                mask = ids.ne(self.tokenizer.pad_token_id)\r\n                hidden = self.model(input_ids=ids, attention_mask=mask).last_hidden_state\r\n                vectors = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)\r\n                output.append(vectors.float())\r\n        return torch.cat(output)\r\n\r\n    def rank(self, query: str, candidates: list[CodeBlock], topk: int):\r\n        import torch\r\n        if not candidates:\r\n            return [], []\r\n        q = self.encode([query], True)\r\n        docs = self.encode([render_block(x) for x in candidates], False)\r\n        if self.scorer == "cosine":\r\n            scores = torch.nn.functional.cosine_similarity(q[:, None], docs[None], dim=-1)[0]\r\n        elif self.scorer in ("qiepsm_fidelity", "qiepsm_log_fidelity", "qiepsm_learned_phase"):\r\n            from controlled_comparison import QIEPSMFidelity\r\n            with torch.inference_mode():\r\n                if self.scorer == \'qiepsm_learned_phase\':\r\n                    head = self.head\r\n                else:\r\n                    head = QIEPSMFidelity()\r\n                score = head.pairwise_log_fidelity if self.scorer in ("qiepsm_log_fidelity", "qiepsm_learned_phase") else head.pairwise_fidelity\r\n                scores = score(q.float(), docs.float()[None])[0]\r\n        else:\r\n            scores = self.head.score(q, docs)\r\n        order = torch.argsort(scores, descending=True, stable=True)[:topk].tolist()\r\n        return [candidates[i] for i in order], [float(scores[i]) for i in order]\r\n\r\n\r\ndef discover_checkpoints(args) -> dict[str, Path]:\r\n    base = Path(args.checkpoint_root) if args.checkpoint_root else Path("/kaggle/input")\r\n    if args.methods in (["qiepsm_fidelity"], ["qiepsm_log_fidelity"], ["qiepsm_learned_phase"]):\r\n        if not (base / "model.safetensors").is_file() or not (base / "config.json").is_file():\r\n            raise FileNotFoundError(f"Explicit QIEPSM encoder checkpoint missing: {base}")\r\n        return {args.methods[0]: base}\r\n    # Kaggle has used both /kaggle/input/<slug> and namespaced input layouts.\r\n    # Search recursively from the input root instead of assuming one mount path.\r\n    search_bases = [base]\r\n    kaggle_input = Path("/kaggle/input")\r\n    try:\r\n        base.relative_to(kaggle_input)\r\n    except ValueError:\r\n        pass\r\n    else:\r\n        if base != kaggle_input:\r\n            search_bases.append(kaggle_input)\r\n\r\n    found = {}\r\n    inspected_files = []\r\n    for search_base in dict.fromkeys(search_bases):\r\n        if not search_base.exists():\r\n            continue\r\n        inspected_files.extend(str(path) for path in search_base.rglob("*") if path.is_file())\r\n        roots = [search_base, *(path for path in search_base.rglob("*") if path.is_dir())]\r\n        for path in roots:\r\n            if not (path / "model.safetensors").is_file() or not (path / "config.json").is_file():\r\n                continue\r\n            if (path / "quantum_head.safetensors").is_file():\r\n                found["quantum"] = path\r\n            elif "cosine" in str(path).lower():\r\n                found["cosine"] = path\r\n    missing = set(args.methods) - set(found)\r\n    if missing:\r\n        preview = inspected_files[:80]\r\n        raise FileNotFoundError(\r\n            f"Checkpoint directories not found for {sorted(missing)}; found={found}; "\r\n            f"searched={[str(x) for x in search_bases]}; input_files={preview}")\r\n    print("Checkpoint directories:", {name: str(path) for name, path in found.items()}, flush=True)\r\n    return found\r\n\r\n\r\ndef retrieval_phase(args, rows, out, checkpoints):\r\n    import torch\r\n    path, done = out / "retrieval.jsonl", read_jsonl(out / "retrieval.jsonl")\r\n    for method in args.methods:\r\n        pending = [r for r in rows if f"{method}/{r[\'key\']}" not in done]\r\n        if not pending:\r\n            continue\r\n        retriever = RLCoderRetriever(checkpoints[method], method, args)\r\n        for index, row in enumerate(pending, 1):\r\n            example = make_example(row)\r\n            pool = bm25_pool(example, args.bm25_topk)\r\n            pool.append(CodeBlock("", STOP_TEXT, "", example.language, ""))\r\n            started = time.perf_counter()\r\n            chosen, scores = retriever.rank(query_text(example), pool, args.top_k)\r\n            key = f"{method}/{row[\'key\']}"\r\n            record = {"key": key, "task_id": row["task_id"], "language": row["language"],\r\n                      "method": method, "query": query_text(example), "bm25_candidates": len(pool) - 1,\r\n                      "candidates": [block_dict(x) for x in chosen], "scores": scores,\r\n                      "stop_rank": next((i + 1 for i, x in enumerate(chosen) if x.file_path == ""), None),\r\n                      "seconds": time.perf_counter() - started}\r\n            append_jsonl(path, record); done[key] = record\r\n            if index % 100 == 0 or index == len(pending):\r\n                print(f"Retrieval {method} {index}/{len(pending)}", flush=True)\r\n        del retriever\r\n        gc.collect()\r\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\r\n    return done\r\n\r\n\r\ndef render_prompt(tokenizer, example: Example, blocks: list[CodeBlock], args):\r\n    kept = []\r\n    for block in blocks:\r\n        if block.file_path == "":\r\n            break\r\n        kept.append(block)\r\n    cross = "\\n\\n".join(render_block(x) for x in kept)\r\n    cross_ids = tokenizer.encode(cross[:args.cross_tokens * 10], add_special_tokens=False)[:args.cross_tokens]\r\n    marker = "#" if example.language == "python" else "//"\r\n    path_ids = tokenizer.encode(f"\\n\\n{marker} file path: {example.file_path}\\n\\n", add_special_tokens=False)\r\n    allowed = args.context_tokens - len(cross_ids) - len(path_ids) - 10\r\n    if allowed <= 0: raise ValueError("Cross-file context exhausted prompt budget")\r\n    local_ids = tokenizer.encode(example.left_context, add_special_tokens=False)[-allowed:]\r\n    text = tokenizer.decode(cross_ids + path_ids + local_ids)\r\n    ids = tokenizer.encode(text)[-args.context_tokens:]\r\n    return ids, {"cross_tokens": len(cross_ids), "local_tokens": len(local_ids),\r\n                 "prompt_tokens": len(ids), "prompt_sha256": digest(ids)}\r\n\r\n\r\ndef generation_phase(args, rows, out, ranked):\r\n    import torch\r\n    from transformers import AutoModelForCausalLM, AutoTokenizer\r\n    path, done = out / "predictions.jsonl", read_jsonl(out / "predictions.jsonl")\r\n    pending = [(m, r) for r in rows for m in args.methods if f"{m}/{r[\'key\']}" not in done]\r\n    if not pending: return done\r\n    tokenizer = AutoTokenizer.from_pretrained(args.generator, revision=args.generator_revision)\r\n    if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token\r\n    tokenizer.padding_side = "left"\r\n    dtype = torch.float16 if args.device.startswith("cuda") else torch.float32\r\n    model = AutoModelForCausalLM.from_pretrained(args.generator, revision=args.generator_revision,\r\n              torch_dtype=dtype, attn_implementation="sdpa").to(args.device).eval()\r\n    for start in range(0, len(pending), args.generation_batch_size):\r\n        items = pending[start:start + args.generation_batch_size]\r\n        prompts, infos = [], []\r\n        for method, row in items:\r\n            example = make_example(row)\r\n            blocks = [dict_block(x) for x in ranked[f"{method}/{row[\'key\']}"]["candidates"]]\r\n            ids, info = render_prompt(tokenizer, example, blocks, args)\r\n            prompts.append(ids); infos.append(info)\r\n        batch = tokenizer.pad({"input_ids": prompts}, padding=True, return_tensors="pt").to(args.device)\r\n        input_length = batch.input_ids.shape[1]\r\n        if args.device.startswith("cuda"): torch.cuda.synchronize()\r\n        started = time.perf_counter()\r\n        with torch.inference_mode():\r\n            generated = model.generate(**batch, max_new_tokens=args.max_new_tokens, do_sample=False,\r\n                                       num_beams=1, pad_token_id=tokenizer.pad_token_id,\r\n                                       eos_token_id=tokenizer.eos_token_id, use_cache=True)\r\n        if args.device.startswith("cuda"): torch.cuda.synchronize()\r\n        elapsed = time.perf_counter() - started\r\n        for i, ((method, row), info) in enumerate(zip(items, infos)):\r\n            tokens = generated[i, input_length:].tolist()\r\n            key = f"{method}/{row[\'key\']}"\r\n            record = {"key": key, "task_id": row["task_id"], "language": row["language"],\r\n                      "method": method, "raw_prediction": tokenizer.decode(tokens, skip_special_tokens=True),\r\n                      "generated_tokens": len(tokens), "hit_token_limit": len(tokens) >= args.max_new_tokens,\r\n                      "batch_seconds": elapsed, **info}\r\n            append_jsonl(path, record); done[key] = record\r\n        completed = min(start + len(items), len(pending))\r\n        if completed // 100 > start // 100 or completed == len(pending):\r\n            print(f"Completion {completed}/{len(pending)}", flush=True)\r\n    return done\r\n\r\n\r\ndef make_parser(language):\r\n    from tree_sitter import Language, Parser\r\n    if language == "python":\r\n        import tree_sitter_python as grammar\r\n    else:\r\n        import tree_sitter_java as grammar\r\n    metrics.get_ast = getattr(metrics.get_ast, "__wrapped__", metrics.get_ast)\r\n    return Parser(Language(grammar.language()))\r\n\r\n\r\ndef score_prediction(prefix, raw, target, language, parser):\r\n    pred = metrics.remove_comments(metrics.postprocess_code_lines(prefix, raw, parser, language))\r\n    target = metrics.remove_comments(target)\r\n    pred_lines = [x.strip() for x in pred.splitlines() if x.strip()]\r\n    target_lines = [x.strip() for x in target.splitlines() if x.strip()]\r\n    return {"prediction_processed": pred, "target_processed": target,\r\n            "em": 100.0 * (pred_lines == target_lines),\r\n            "es": float(metrics.cal_edit_sim([target], [pred]))}\r\n\r\n\r\ndef evaluate(args, rows, out, predictions):\r\n    details, summary = [], []\r\n    parsers = {language: make_parser(language) for language in args.languages}\r\n    for language in args.languages:\r\n        subset = [x for x in rows if x["language"] == language]\r\n        for method in args.methods:\r\n            group = []\r\n            for row in subset:\r\n                key = f"{method}/{row[\'key\']}"\r\n                if key not in predictions: raise ValueError(f"Missing prediction {key}")\r\n                score = score_prediction(row["left_context"], predictions[key]["raw_prediction"],\r\n                                         row["groundtruth"], language, parsers[language])\r\n                item = {"key": key, "task_id": row["task_id"], "language": language,\r\n                        "method": method, **score}\r\n                group.append(item); details.append(item)\r\n            summary.append({"language": language, "method": method, "n": len(group),\r\n                            "em": float(np.mean([x["em"] for x in group])),\r\n                            "es": float(np.mean([x["es"] for x in group]))})\r\n    write_json(out / "metrics.json", {"status": "complete", "scale": "0-100", "results": summary})\r\n    pd.DataFrame(summary).to_csv(out / "metrics.csv", index=False)\r\n    with (out / "scored_predictions.jsonl").open("w", encoding="utf-8") as stream:\r\n        for item in details: stream.write(json.dumps(item, ensure_ascii=False) + "\\n")\r\n    print(pd.DataFrame(summary).to_string(index=False))\r\n\r\n\r\ndef main():\r\n    p = argparse.ArgumentParser(description=__doc__)\r\n    p.add_argument("--output", default="/kaggle/working/rlcoder-eval-results")\r\n    p.add_argument("--checkpoint-root", default=None)\r\n    p.add_argument("--validation-jsonl", default=None)\r\n    p.add_argument("--data-root", default=None)\r\n    p.add_argument("--dataset-revision", default=DATASET_REVISION)\r\n    p.add_argument("--languages", nargs="+", choices=["python", "java"], default=["python", "java"])\r\n    p.add_argument("--methods", nargs="+", choices=["cosine", "quantum", "qiepsm_fidelity", "qiepsm_log_fidelity", "qiepsm_learned_phase"], default=["cosine", "quantum"])\r\n    p.add_argument("--limit", type=int, default=0)\r\n    p.add_argument("--seed", type=int, default=42)\r\n    p.add_argument("--generator", default=GENERATOR)\r\n    p.add_argument("--generator-revision", default=GENERATOR_REVISION)\r\n    p.add_argument("--bm25-topk", type=int, default=100)\r\n    p.add_argument("--top-k", type=int, default=10)\r\n    p.add_argument("--query-tokens", type=int, default=256)\r\n    p.add_argument("--candidate-tokens", type=int, default=256)\r\n    p.add_argument("--embedding-batch-size", type=int, default=32)\r\n    p.add_argument("--generation-batch-size", type=int, default=6)\r\n    p.add_argument("--context-tokens", type=int, default=2048)\r\n    p.add_argument("--cross-tokens", type=int, default=1536)\r\n    p.add_argument("--max-new-tokens", type=int, default=64)\r\n    p.add_argument("--n-qubits", type=int, default=32)\r\n    p.add_argument("--device", default="cuda:0")\r\n    p.add_argument("--evaluate-only", action="store_true")\r\n    args = p.parse_args()\r\n    if args.limit < 0 or min(args.bm25_topk, args.top_k, args.embedding_batch_size,\r\n                             args.generation_batch_size) < 1: p.error("Invalid positive count")\r\n    out = Path(args.output); out.mkdir(parents=True, exist_ok=True)\r\n    rows, sources = prepare_data(args)\r\n    checkpoints = discover_checkpoints(args)\r\n    identity = {"schema": SCHEMA_VERSION,\r\n                "configuration": {k: v for k, v in vars(args).items() if k not in {"output", "evaluate_only", "data_root"}},\r\n                "sources": sources,\r\n                "checkpoints": {k: {"path": str(v), "model_sha256": file_hash(v / "model.safetensors"),\r\n                                     **({"phase_sha256": file_hash(v / "phase_head.safetensors")} if k == "qiepsm_learned_phase" else {}),\r\n                                     **({"head_sha256": file_hash(v / "quantum_head.safetensors")} if k == "quantum" else {})}\r\n                                for k, v in checkpoints.items()}}\r\n    manifest = out / "manifest.json"\r\n    if manifest.exists() and json.loads(manifest.read_text(encoding="utf-8"))["identity_hash"] != digest(identity):\r\n        raise ValueError("Configuration changed; choose a new output directory")\r\n    write_json(manifest, {"identity_hash": digest(identity), **identity})\r\n    if args.evaluate_only:\r\n        predictions = read_jsonl(out / "predictions.jsonl")\r\n    else:\r\n        import torch\r\n        if args.device.startswith("cuda") and not torch.cuda.is_available():\r\n            raise RuntimeError("Enable a Kaggle GPU")\r\n        ranked = retrieval_phase(args, rows, out, checkpoints)\r\n        predictions = generation_phase(args, rows, out, ranked)\r\n    evaluate(args, rows, out, predictions)\r\n\r\n\r\nif __name__ == "__main__":\r\n    main()\r\n')
p = ENGINE / "generation_eval" / 'rlcoder_compat.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_bytes(b'"""Extracted from local RLCoder source; regenerate with build_rlcoder_compat.py.\r\n\r\nCodeBlock, Example, sampling, natural block splitting and generator prompt/dataset\r\nare upstream implementations. Added bounded sampling attempts only. Identifier\r\npredicate originates in RLCoder/utils/eval_utils.py (Apache-2.0, Amazon.com).\r\nLanguage keyword handling follows Microsoft\'s MIT-licensed keywordlist.py.\r\n"""\r\nimport random\r\nimport re\r\nimport keyword\r\nimport torch\r\nimport pandas as pd\r\nfrom torch.utils.data import Dataset\r\nIDENTIFIER_REGEX = re.compile(\'[_a-zA-Z][_a-zA-Z0-9]*\')\r\nSOURCE_SHA256 = {\'datasets.py\': \'14348d392f60e71dc48fde2423abdf200baa6f5065af5d998d169a92278f20e7\', \'bm25.py\': \'dd3ecb482a1f3dafd4fcc98f606a287d8e891c7a189c27e7d566c8a750e3cc4e\', \'generator.py\': \'83911d18ac851b126dfef0ed9d7cb514ede028e07b08063d9ff342c5ef45f1a5\', \'utils/eval_utils.py\': \'ba5b78e79bac166b0ee6f4d7d458ff5516afc0acd5a33b0da345f95c87eb0f1a\', \'utils/keywords/java.txt\': \'807f36a9985ec3e75052b21ec3a8933f883deae84aa9cdac6c4779005e08752f\'}\r\nJAVA_KEYWORDS = frozenset([\'abstract\', \'assert\', \'boolean\', \'break\', \'byte\', \'case\', \'catch\', \'char\', \'class\', \'continue\', \'default\', \'do\', \'double\', \'else\', \'enum\', \'extends\', \'final\', \'finally\', \'float\', \'for\', \'if\', \'implements\', \'import\', \'instanceof\', \'int\', \'interface\', \'long\', \'native\', \'new\', \'package\', \'private\', \'protected\', \'public\', \'return\', \'short\', \'static\', \'strictfp\', \'super\', \'switch\', \'synchronized\', \'this\', \'throw\', \'throws\', \'transient\', \'try\', \'void\', \'volatile\', \'while\', \'var\', \'const\', \'goto\'])\r\n\r\ndef get_language_keywords(lang):\r\n    if lang == \'python\':\r\n        return frozenset(k for k in keyword.kwlist if k not in (\'True\', \'False\'))\r\n    if lang == \'java\':\r\n        return JAVA_KEYWORDS\r\n    raise ValueError(lang)\r\n\r\nclass CodeBlock(object):\r\n    def __init__(self, file_path, description, code_content, language, _type):\r\n        """\r\n        Represents a block of code.\r\n        :param file_path: The path to the code file.\r\n        :param description: The description of the code block.\r\n        :param code_content: The content of the code block.\r\n        """\r\n        self.file_path = file_path\r\n        self.code_content = code_content\r\n        self.description = description\r\n        self.language = language\r\n        self._type = _type\r\n\r\n    def __str__(self):\r\n        if self.language == "python":\r\n            comment_label = "#"\r\n        else:\r\n            comment_label = "//"\r\n        crossfile_context = "\\n".join([f"{comment_label} {cl}" for cl in  self.description.strip().split(\'\\n\') if cl]) + "\\n"\r\n        crossfile_context += "\\n".join([f"{comment_label} {cl}" for cl in  self.code_content.split(\'\\n\') if cl])\r\n        return crossfile_context.strip()\r\n\r\nclass Example(object):\r\n    def __init__(self, task_id, file_path, left_context, right_context, related_files, target_code, language):\r\n        """\r\n        Represents an example used for constructing a dataset.\r\n        :param task_id: Task ID.\r\n        :param file_path: File path.\r\n        :param left_context: The context to the left of the target code.\r\n        :param right_context: The context to the right of the target code.\r\n        :param related_files: A list of related files, each containing a path and text.\r\n        :param target_code: The target code snippet.\r\n        """\r\n        self.task_id = task_id\r\n        self.file_path = file_path\r\n        self.left_context = left_context\r\n        self.right_context = right_context\r\n        self.related_files = related_files\r\n        self.target_code = target_code\r\n        self.language = language\r\n\r\n    def __str__(self):\r\n        return (f"[Example]:\\n"\r\n                f"[Task ID]:\\n{self.task_id}\\n"\r\n                f"[Path]:\\n{self.file_path}\\n"\r\n                f"[Left Context]:\\n{self.left_context}\\n"\r\n                f"[Target Code]:\\n{self.target_code}\\n"\r\n                f"[Right Context]:\\n{self.right_context}\\n"\r\n                f"[Related Files]:\\n{len(self.related_files)} files\\n"\r\n        )\r\n\r\ndef load_train_and_valid_dataset(data_root):\r\n    """\r\n    Loads the training dataset.\r\n    :return: The training dataset.\r\n    """\r\n    training_datasets = []\r\n    validation_datasets = []\r\n    for language in ["python", "java"]:\r\n        data_frame = pd.read_parquet(f"{data_root}/github_repos/{language}/train.parquet")\r\n        all_data = []\r\n        temp_data = []\r\n        for x in data_frame[["path", "content", "first"]].values:\r\n            if x[-1]:  # At the start of a new file\r\n                if len(temp_data) > 1:\r\n                    all_data.append((temp_data,language))\r\n                temp_data = []\r\n            temp_data.append([x[0], x[1]])\r\n        training_datasets.extend(all_data[:2000])\r\n        validation_datasets.extend(all_data[2000:2200])\r\n    random.shuffle(training_datasets)\r\n    random.shuffle(validation_datasets)\r\n\r\n    return training_datasets, validation_datasets\r\n\r\ndef construct_dataset(raw_data, num_samples):\r\n    """\r\n    Builds a dataset.\r\n    :param raw_data: Raw data.\r\n    :param num_samples: The number of samples to generate.\r\n    :return: The list of constructed samples.\r\n    """\r\n    examples = []\r\n    data_index = 0\r\n    while len(examples) < num_samples:\r\n        example,language = raw_data[data_index % len(raw_data)]\r\n        data_index += 1\r\n        if data_index > max(1000, num_samples * 100):\r\n            raise RuntimeError("Insufficient eligible samples within bounded attempt budget")\r\n        selected_file = random.choice(example[1:])\r\n        related_files = [CodeBlock(x[0], f"file path: {x[0]}\\nlines: {0}-{len(x[1].splitlines())}", x[1], language, \'\') for x in example if x[0] != selected_file[0]]\r\n        path = selected_file[0]\r\n        selected_file_content = selected_file[1].split(" ")\r\n        try_count = 0\r\n\r\n        while try_count < 10:\r\n            end_line_number = int(len(selected_file_content) * random.uniform(0.2, 0.8))\r\n            left_context = " ".join(selected_file_content[:end_line_number])\r\n            target_length = random.randint(32, 64)\r\n            target = " ".join(selected_file_content[end_line_number:end_line_number + target_length])\r\n            right_context = " ".join(selected_file_content[end_line_number + target_length:])\r\n            if len(left_context.split()) > 80 and len(target.split()) > 8:\r\n                examples.append(\r\n                    Example(len(examples), path, left_context, right_context, related_files, target,language)\r\n                )\r\n                break\r\n            # if language == \'python\':\r\n            #     if len(left_context.split()) > 64 and len(target.split()) > 5:\r\n            #         examples.append(\r\n            #             Example(len(examples), path, left_context, right_context, related_files, target,language)\r\n            #         )\r\n            #         break\r\n            # elif language == \'java\':\r\n            #     if len(left_context.split()) > 80 and len(target.split()) > 8:\r\n            #         examples.append(\r\n            #             Example(len(examples), path, left_context, right_context, related_files, target,language)\r\n            #         )\r\n            #         break\r\n            try_count += 1\r\n    \r\n    return examples\r\n\r\ndef split_into_smaller_blocks(code_block, enable_fixed_block):\r\n    """\r\n    Split large blocks of code into smaller ones, each containing no more than 12 non-empty lines.\r\n    """\r\n    smaller_blocks = []\r\n\r\n    if enable_fixed_block:\r\n        lines = [line for line in code_block.code_content.split(\'\\n\') if line.strip() != \'\']\r\n        for i in range(0, min(len(lines),5000), 12):\r\n            start_line_offset = i\r\n            end_line_offset = min(i + 12, len(lines))\r\n            block_content = \'\\n\'.join(lines[start_line_offset:end_line_offset])\r\n            smaller_blocks.append(CodeBlock(code_block.file_path, \r\n                                            f"file path: {code_block.file_path}\\nlines: {start_line_offset}-{end_line_offset - 1}",\r\n                                            block_content,\r\n                                            code_block.language,\r\n                                            \'fixed_block\'))\r\n\r\n    else:\r\n        # Split the code by spaces, then reassemble it into blocks.\r\n        mini_blocks = []\r\n        current_block = [] \r\n        for line in code_block.code_content.splitlines(): \r\n            if line.strip() == \'\':  \r\n                if current_block: \r\n                    mini_blocks.append(current_block)\r\n                    current_block = []\r\n            else:\r\n                current_block.append(line)\r\n        if current_block: \r\n            mini_blocks.append(current_block)\r\n\r\n        max_len = 15\r\n        temp_mini_blocks = []\r\n        for mini_block in mini_blocks:\r\n            if len(mini_block) > max_len:\r\n                for idx in range(0, len(mini_block), max_len):\r\n                    temp_mini_blocks.append(mini_block[idx: idx+max_len])\r\n            else:\r\n                temp_mini_blocks.append(mini_block)\r\n        mini_blocks = temp_mini_blocks\r\n\r\n        current_content = []\r\n        total_lines = 0  \r\n        for block in mini_blocks:\r\n            if total_lines >= 5000:  \r\n                break  \r\n            if len(current_content) + len(block) <= 15:  \r\n                current_content.extend(block)\r\n                total_lines += len(block)  \r\n            else:  \r\n                if current_content:  \r\n                    smaller_blocks.append(CodeBlock(code_block.file_path, \r\n                                                    f"file path: {code_block.file_path}\\nlines: {total_lines-len(current_content)+1}-{total_lines}",\r\n                                                    \'\\n\'.join(current_content),\r\n                                                    code_block.language,\r\n                                                    \'mini_block\'))\r\n                current_content = block  \r\n                total_lines += len(block)  \r\n        if current_content:  \r\n            smaller_blocks.append(CodeBlock(code_block.file_path, \r\n                                            f"file path: {code_block.file_path}\\nlines: {total_lines-len(current_content)+1}-{total_lines}",\r\n                                            \'\\n\'.join(current_content),\r\n                                            code_block.language,\r\n                                            \'mini_block\'))\r\n        \r\n    return smaller_blocks\r\n\r\nclass GeneratorDataset(Dataset):\r\n    """\r\n    A dataset class for code generation.\r\n\r\n    Args:\r\n        args: Configuration parameters.\r\n        tokenizer: Tokenizer.\r\n        examples: A collection of examples.\r\n        retrieved_codeblocks: Retrieved code blocks.\r\n    """\r\n    def __init__(self, args, tokenizer, examples, retrieved_codeblocks, generation=False):\r\n        self.args = args\r\n        self.tokenizer = tokenizer\r\n        self.examples = examples\r\n        self.retrieved_codeblocks = retrieved_codeblocks\r\n        self.generation = generation\r\n\r\n    def __len__(self):\r\n        return len(self.examples)\r\n\r\n    def construct_prompts(self,example,retrieved_codeblocks):\r\n        filter_codeblocks = []\r\n        for x in retrieved_codeblocks:\r\n            # Keep only the blocks before the empty block\r\n            if x.file_path != "":\r\n                filter_codeblocks.append(x)\r\n            else:\r\n                break\r\n        crossfile_context = "\\n\\n".join([str(retrieved_codeblock) for retrieved_codeblock in filter_codeblocks])\r\n        crossfile_context = self.tokenizer.encode(crossfile_context[:self.args.generator_max_crossfile_length*10], add_special_tokens=False)[:self.args.generator_max_crossfile_length]\r\n        path_context = f"\\n\\n# file path: {example.file_path}\\n\\n"\r\n        path_context = self.tokenizer.encode(path_context, add_special_tokens=False)\r\n        allowed_prompt_length = self.args.generator_max_context_length - (len(crossfile_context)+len(path_context)+10)\r\n        infile_context = self.tokenizer.encode(example.left_context, add_special_tokens=False)[-allowed_prompt_length:]\r\n\r\n        prompt = self.tokenizer.decode(crossfile_context + path_context + infile_context)\r\n        return prompt\r\n\r\n    def __getitem__(self, idx):\r\n        example = self.examples[idx]\r\n        retrieved_codeblocks = self.retrieved_codeblocks[idx]\r\n        prompt = self.construct_prompts(example,retrieved_codeblocks)\r\n        \r\n        prompt_ids = self.tokenizer.encode(prompt)[-self.args.generator_max_context_length:]\r\n        if self.generation:\r\n             padding_length = self.args.generator_max_context_length - len(prompt_ids)\r\n             input_ids = [self.tokenizer.pad_token_id] * padding_length + prompt_ids\r\n             return torch.tensor(input_ids)\r\n\r\n        target_ids = self.tokenizer.encode(example.target_code, add_special_tokens=False)[:self.args.generator_max_generation_length]\r\n\r\n        input_ids = prompt_ids + target_ids\r\n        labels = [-100 for _ in prompt_ids] + target_ids\r\n\r\n        padding_length = self.args.generator_max_context_length + self.args.generator_max_generation_length - len(input_ids)\r\n        input_ids = [self.tokenizer.pad_token_id] * padding_length + input_ids\r\n        labels = [-100] * padding_length + labels \r\n\r\n        return torch.tensor(input_ids), torch.tensor(labels)\r\n\r\ndef is_identifier(token, lang=None):\r\n    return True if IDENTIFIER_REGEX.match(token) \\\r\n                   and (lang is None or token not in get_language_keywords(lang)) \\\r\n        else False\r\n')
p = ENGINE / "generation_eval" / 'requirements.txt'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_bytes(b"# Keep Kaggle's CUDA-enabled torch.\ntransformers==4.51.3\naccelerate==1.6.0\nhuggingface-hub==0.30.2\nsafetensors>=0.5.3\npandas>=2.2,<3\npyarrow>=17\nrank-bm25==0.2.2\ntree-sitter==0.23.2\ntree-sitter-python==0.23.6\ntree-sitter-java>=0.23.2\nfuzzywuzzy==0.18.0\npython-Levenshtein==0.27.1\ntimeout-decorator==0.5.0\nsacrebleu==2.5.1\nnltk>=3.8,<4\n")
p = ENGINE / "generation_eval" / 'vendor_aligncoder/__init__.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_bytes(b'\r\n')
p = ENGINE / "generation_eval" / 'vendor_aligncoder/utils/eval_utils.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_bytes(b'# Copyright Amazon.com, Inc. or its affiliates. All rights reserved.\r\n# Licensed under the Apache License, Version 2.0 (the "License");\r\n# you may not use this file except in compliance with the License.\r\n# You may obtain a copy of the License at\r\n#\r\n#     http://www.apache.org/licenses/LICENSE-2.0\r\n#\r\n# Unless required by applicable law or agreed to in writing, software\r\n# distributed under the License is distributed on an "AS IS" BASIS,\r\n# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\r\n# See the License for the specific language governing permissions and\r\n# limitations under the License.\r\n\r\nimport re\r\nimport ast\r\nimport torch\r\nfrom typing import List\r\nimport timeout_decorator\r\nfrom fuzzywuzzy import fuzz\r\nfrom functools import lru_cache\r\nfrom nltk.tokenize import RegexpTokenizer\r\nfrom .keywords.keywordlist import get_language_keywords\nfrom sacrebleu.tokenizers.tokenizer_intl import TokenizerV14International\r\n\r\nIDENTIFIER_REGEX = re.compile(\'[_a-zA-Z][_a-zA-Z0-9]*\')\r\nREGEX_TEXT = ("(?<=[a-z0-9])(?=[A-Z])|"\r\n              "(?<=[A-Z0-9])(?=[A-Z][a-z])|"\r\n              "(?<=[0-9])(?=[a-zA-Z])|"\r\n              "(?<=[A-Za-z])(?=[0-9])|"\r\n              "(?<=[@$.\'\\"])(?=[a-zA-Z0-9])|"\r\n              "(?<=[a-zA-Z0-9])(?=[@$.\'\\"])|"\r\n              "_|\\\\s+")\r\nstring_pattern = r\'"([^"\\\\]*(\\\\.[^"\\\\]*)*)"|\\\'([^\\\'\\\\]*(\\\\.[^\\\'\\\\]*)*)\\\'\'\r\n\r\nSPLIT_REGEX = re.compile(REGEX_TEXT)\r\n\r\nstr_tokenizer = TokenizerV14International()\r\ncode_tokenizer = RegexpTokenizer(r\'\\w+\')\r\n\r\n\r\ndef cal_edit_sim(references, hypotheses):\r\n    total = len(references)\r\n    edit_sim = 0.0\r\n    for pred, gt in zip(hypotheses, references):\r\n        pred = pred.strip()\r\n        gt = gt.strip()\r\n        edit_sim += fuzz.ratio(pred, gt)\r\n    return edit_sim / total\r\n\r\n\r\n@lru_cache(maxsize=5000)\r\ndef split_identifier_into_parts(identifier: str) -> List[str]:\r\n    """\r\n    Split a single identifier into parts on snake_case and camelCase\r\n    """\r\n    identifier_parts = list(s for s in SPLIT_REGEX.split(identifier) if len(s) > 0)\r\n\r\n    if len(identifier_parts) == 0:\r\n        return [identifier]\r\n    if "_" in identifier:  # We consider "_" as part of identifier and add it back in between each semantic part\r\n        # if snake_case, we only split identifiers based on "_", ignore the mixed camelCase or other special symbols\r\n        # this helps us avoid splitting identifiers like "get_2d_array" into ["get", "2", "d", "array"]\r\n        # also avoid many other corner cases\r\n        identifier_parts = identifier.split("_")\r\n        tmp = [identifier_parts[0]]\r\n        for i in identifier_parts[1:]:\r\n            tmp.append("_")\r\n            tmp.append(i)\r\n        identifier_parts = tmp\r\n\r\n    return identifier_parts\r\n\r\n\r\ndef is_identifier(token, lang=None):\r\n    return IDENTIFIER_REGEX.match(token) and (lang is None or token not in get_language_keywords(lang))\r\n\r\n\r\ndef extract_identifiers(source_code, lang):\r\n    # the main idea is to remove String from a source code\r\n    # then, tokenize the code to get all words and match with identifier regular expression\r\n    # check if it is a language specific keyword, it not, then it is an identifier\r\n    source_code_without_strings = re.sub(string_pattern, \'\', source_code)\r\n    _ids = [t for t in code_tokenizer.tokenize(source_code_without_strings) if is_identifier(t, lang)]\r\n    return _ids\r\n\r\n\r\ndef tokenize_string(input_str):\r\n    return str_tokenizer(input_str)\r\n\r\n\r\ndef get_bracket_lang_statement(completion):\r\n    end_idx = None\r\n    for i in range(len(completion)):\r\n        if completion[i] in [";", "}", "{"]:\r\n            end_idx = i\r\n            break\r\n    return completion[:end_idx + 1] if end_idx else completion\r\n\r\n\r\n@timeout_decorator.timeout(5)\r\ndef get_ast(parser, code):\r\n    assert isinstance(code, str) or isinstance(code, bytes)\r\n    if isinstance(code, str):\r\n        code = bytes(code, "utf8")\r\n    try:\r\n        tree = parser.parse(code)\r\n        return tree\r\n    except Exception as e:\r\n        return None\r\n\r\n\r\ndef remove_comments(code):\r\n    code = re.sub(r\'#.*\', \'\', code)\r\n    code = re.sub(r\'//.*\', \'\', code)\r\n    return code\r\n\r\n\r\ndef is_parse_valid(parser, code):\r\n    def syntax_error(node):\r\n        if node.type == "ERROR":\r\n            return True\r\n        try:\r\n            for child in node.children:\r\n                if syntax_error(child):\r\n                    return True\r\n        except RecursionError as err:\r\n            return True\r\n\r\n        return False\r\n\r\n    tree = get_ast(parser, code)\r\n    if tree is not None:\r\n        return not syntax_error(tree.root_node)\r\n    return False\r\n\r\n\r\ndef is_code_parseable(code):\r\n    try:\r\n        ast.parse(code)\r\n        return True\r\n    except SyntaxError:\r\n        return False\r\n\r\n\r\ndef get_python_one_statement(prompt, completion, parser):\r\n    for i in range(len(completion)):\r\n        code = prompt + completion[:i + 1]\r\n        if not is_parse_valid(parser, code):\r\n            continue\r\n        if completion[i + 1] == "\\n":\r\n            return completion[:i + 1].rstrip()\r\n    return completion\r\n\r\n\r\ndef postprocess_code_lines(prompt, completion, parser, lang):\r\n    try:\r\n        if lang in ["java", "csharp", "typescript"]:\r\n            return get_bracket_lang_statement(completion)\r\n        elif lang == "python":\r\n            return get_python_one_statement(prompt, completion, parser)\r\n    except Exception as e:\r\n        return completion\r\n\r\n\r\ndef compute_mean_logp(scores, sequences, pad_token_id):\r\n    assert scores.shape[0] == sequences.shape[0]\r\n    assert scores.shape[1] == sequences.shape[1]\r\n    with torch.no_grad():\r\n        logp_vocab = torch.nn.functional.log_softmax(scores, dim=-1)\r\n        indices = torch.unsqueeze(sequences, dim=-1)\r\n        logp = torch.gather(logp_vocab, dim=-1, index=indices).squeeze(-1)\r\n        sum_logp = torch.cumsum(logp, dim=1)  # batch_size, seq_len\r\n        denom = torch.arange(1, sum_logp.shape[1] + 1).reshape(1, -1).to(device=sum_logp.device)  # 1, seq_len\r\n        mean_logp = (sum_logp / denom).tolist()  # batch_size, seq_len\r\n        sequence_lengths = (sequences != pad_token_id).sum(1).tolist()  # batch_size\r\n        mean_logp = [mean_logp[idx][l - 1] for idx, l in enumerate(sequence_lengths)]\r\n    return mean_logp\r\n')
p = ENGINE / "generation_eval" / 'vendor_aligncoder/utils/__init__.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_bytes(b'')
p = ENGINE / "generation_eval" / 'vendor_aligncoder/utils/keywords/csharp.txt'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_bytes(b'abstract\r\nas\r\nbase\r\nbool\r\nbreak\r\nbyte\r\ncase\r\ncatch\r\nchar\r\nchecked\r\nclass\r\nconst\r\ncontinue\r\ndecimal\r\ndefault\r\ndelegate\r\ndo\r\ndouble\r\nelse\r\nenum\r\nevent\r\nexplicit\r\nextern\r\nfinally\r\nfixed\r\nfloat\r\nfor\r\nforeach\r\ngoto\r\nif\r\nimplicit\r\nin\r\nint\r\ninterface\r\ninternal\r\nis\r\nlock\r\nlong\r\nnamespace\r\nnew\r\nnull\r\nobject\r\noperator\r\nout\r\noverride\r\nparams\r\nprivate\r\nprotected\r\npublic\r\nreadonly\r\nref\r\nreturn\r\nsbyte\r\nsealed\r\nshort\r\nsizeof\r\nstackalloc\r\nstatic\r\nstring\r\nstruct\r\nswitch\r\nthis\r\nthrow\r\ntry\r\ntypeof\r\nuint\r\nulong\r\nunchecked\r\nunsafe\r\nushort\r\nusing\r\nusing\r\nstatic\r\nvirtual\r\nvoid\r\nvolatile\r\nwhile')
p = ENGINE / "generation_eval" / 'vendor_aligncoder/utils/keywords/java.txt'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_bytes(b'abstract\r\nassert\r\nboolean\r\nbreak\r\nbyte\r\ncase\r\ncatch\r\nchar\r\nclass\r\ncontinue\r\ndefault\r\ndo\r\ndouble\r\nelse\r\nenum\r\nextends\r\nfinal\r\nfinally\r\nfloat\r\nfor\r\nif\r\nimplements\r\nimport\r\ninstanceof\r\nint\r\ninterface\r\nlong\r\nnative\r\nnew\r\npackage\r\nprivate\r\nprotected\r\npublic\r\nreturn\r\nshort\r\nstatic\r\nstrictfp\r\nsuper\r\nswitch\r\nsynchronized\r\nthis\r\nthrow\r\nthrows\r\ntransient\r\ntry\r\nvoid\r\nvolatile\r\nwhile\r\nvar\r\nconst\r\ngoto\r\n')
p = ENGINE / "generation_eval" / 'vendor_aligncoder/utils/keywords/javascript.txt'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_bytes(b'break\r\ncase\r\ncatch\r\nclass\r\nconst\r\ncontinue\r\ndebugger\r\ndefault\r\ndelete\r\ndo\r\nelse\r\nexport\r\nextends\r\nfinally\r\nfor\r\nfunction\r\nif\r\nimport\r\nin\r\ninstanceof\r\nnew\r\nreturn\r\nsuper\r\nswitch\r\nthis\r\nthrow\r\ntry\r\ntypeof\r\nvar\r\nvoid\r\nwhile\r\nwith\r\nyield\r\nenum\r\nimplements\r\ninterface\r\nlet\r\npackage\r\nprivate\r\nprotected\r\npublic\r\nstatic')
p = ENGINE / "generation_eval" / 'vendor_aligncoder/utils/keywords/keywordlist.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_bytes(b'# Original Copyright 2021 Microsoft under MIT License.\r\n# From https://github.com/microsoft/dpu-utils/blob/master/python/dpu_utils/codeutils/keywords/keywordlist.py\r\n\r\nimport os\r\nimport keyword\r\nfrom functools import lru_cache\r\nfrom typing import FrozenSet\r\n\r\n__all__ = [\'get_language_keywords\']\r\n\r\n_LANGUAGE_TO_FILENAME = {\r\n    \'c\': \'c.txt\',\r\n    \'cpp\': \'cpp.txt\',\r\n    \'c++\': \'cpp.txt\',\r\n    \'csharp\': \'csharp.txt\',\r\n    \'c_sharp\': \'csharp.txt\',\r\n    \'c#\': \'csharp.txt\',\r\n    \'go\': \'go.txt\',\r\n    \'java\': \'java.txt\',\r\n    \'javascript\': \'javascript.txt\',\r\n    \'js\': \'javascript.txt\',\r\n    \'php\': \'php.txt\',\r\n    \'ruby\': \'ruby.txt\',\r\n    \'typescript\': \'typescript.txt\',\r\n    \'ts\': \'typescript.txt\',\r\n}\r\n\r\n\r\n@lru_cache()\r\ndef get_language_keywords(language: str) -> FrozenSet[str]:\r\n    """\r\n    Returns the keywords of a programming language.\r\n\r\n    There are some inconsistencies across languages wrt to\r\n    what is considered a keyword. For example, the true/false\r\n    literals are considered keywords in many languages. However,\r\n    we exclude them here for consistency. We also exclude special\r\n    functions-like keywords, such as `die()` in PHP.\r\n    """\r\n    language = language.lower()\r\n    if language == \'python\':\r\n        return frozenset(k for k in keyword.kwlist if k != \'True\' and k != \'False\')\r\n    elif language in _LANGUAGE_TO_FILENAME:\r\n        name = _LANGUAGE_TO_FILENAME[language]\r\n        with open(os.path.join(os.path.dirname(__file__), name)) as f:\r\n            return frozenset(l.strip() for l in f if len(l.strip()) > 0)\r\n    else:\r\n        raise Exception(\'Language keywords `%s` not supported yet. Consider contributing it to dpu-utils.\' % language)\r\n')
p = ENGINE / "generation_eval" / 'vendor_aligncoder/utils/keywords/typescript.txt'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_bytes(b'break\r\ncase\r\ncatch\r\nclass\r\nconst\r\ncontinue\r\ndebugger\r\ndefault\r\ndelete\r\ndo\r\nelse\r\nexport\r\nextends\r\nfinally\r\nfor\r\nfunction\r\nif\r\nimport\r\nin\r\ninstanceof\r\nnew\r\nreturn\r\nsuper\r\nswitch\r\nthis\r\nthrow\r\ntry\r\ntypeof\r\nvar\r\nvoid\r\nwhile\r\nwith\r\nyield\r\nenum\r\nimplements\r\ninterface\r\nlet\r\npackage\r\nprivate\r\nprotected\r\npublic\r\nstatic')
p = ENGINE / "generation_eval" / 'vendor_aligncoder/utils/keywords/__init__.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_bytes(b'')
(ENGINE / "generation_eval" / "controlled_comparison.py").write_text('#!/usr/bin/env python\n"""Controlled cosine-vs-quantum-fidelity experiment for RLCoder.\n\nThe script deliberately separates reward-data preparation from retriever\ntraining.  Both scorers therefore see the exact same queries, candidates, and\nDeepSeek-Coder losses.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport math\nimport os\nimport random\nimport re\nimport time\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom rank_bm25 import BM25Okapi\nfrom torch.optim import AdamW\nfrom torch.utils.data import DataLoader, Dataset\nfrom tqdm.auto import tqdm\nfrom transformers import (\n    AutoModel,\n    AutoModelForCausalLM,\n    AutoTokenizer,\n    get_linear_schedule_with_warmup,\n)\n\n\nNULL_CONTEXT = "Don\'t need cross file context for completion"\n\n\n@dataclass\nclass Candidate:\n    file_path: str\n    description: str\n    code: str\n    language: str\n    is_null: bool = False\n\n    def render(self) -> str:\n        marker = "#" if self.language == "python" else "//"\n        if self.is_null:\n            return ""\n        description = "\\n".join(\n            f"{marker} {line}" for line in self.description.splitlines() if line.strip()\n        )\n        code = "\\n".join(f"{marker} {line}" for line in self.code.splitlines() if line.strip())\n        return f"{description}\\n{code}".strip()\n\n\n@dataclass\nclass CompletionExample:\n    task_id: str\n    file_path: str\n    left_context: str\n    target_code: str\n    language: str\n    related_files: list[tuple[str, str]]\n\n\ndef seed_everything(seed: int) -> None:\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef resolve_data_root(data_root: str) -> Path:\n    requested = Path(data_root)\n    candidates = [\n        requested,\n        requested / "data",\n        Path("/kaggle/input/data4rlcoder/data"),\n        Path("/kaggle/input/data4rlcoder"),\n        Path("data"),\n        Path("RLCoder/data"),\n    ]\n    for candidate in candidates:\n        if (candidate / "github_repos/python/train.parquet").exists():\n            return candidate.resolve()\n    searched = "\\n  - ".join(str(path) for path in candidates)\n    raise FileNotFoundError(\n        "Could not find github_repos/python/train.parquet. Searched:\\n  - " + searched\n    )\n\n\ndef load_repository_groups(data_root: Path) -> tuple[list, list]:\n    train_groups: list[tuple[list[tuple[str, str]], str]] = []\n    valid_groups: list[tuple[list[tuple[str, str]], str]] = []\n    for language in ("python", "java"):\n        frame = pd.read_parquet(data_root / f"github_repos/{language}/train.parquet")\n        groups: list[list[tuple[str, str]]] = []\n        current: list[tuple[str, str]] = []\n        for path, content, first in frame[["path", "content", "first"]].itertuples(index=False, name=None):\n            if first and current:\n                if len(current) > 1:\n                    groups.append(current)\n                current = []\n            current.append((str(path), str(content)))\n        if len(current) > 1:\n            groups.append(current)\n        train_groups.extend((group, language) for group in groups[:2000])\n        valid_groups.extend((group, language) for group in groups[2000:2200])\n    return train_groups, valid_groups\n\n\ndef make_examples(groups: Sequence, count: int, seed: int, split: str) -> list[CompletionExample]:\n    rng = random.Random(seed)\n    order = list(range(len(groups)))\n    rng.shuffle(order)\n    examples: list[CompletionExample] = []\n    attempts = 0\n    max_attempts = max(count * 100, 1000)\n    while len(examples) < count and attempts < max_attempts:\n        group, language = groups[order[attempts % len(order)]]\n        attempts += 1\n        selected_index = rng.randrange(1, len(group))\n        selected_path, selected_content = group[selected_index]\n        tokens = selected_content.split()\n        if len(tokens) < 120:\n            continue\n        cut = int(len(tokens) * rng.uniform(0.2, 0.8))\n        target_length = rng.randint(32, 64)\n        left = " ".join(tokens[:cut])\n        target = " ".join(tokens[cut : cut + target_length])\n        if len(left.split()) <= 80 or len(target.split()) <= 8:\n            continue\n        related = [(path, content) for i, (path, content) in enumerate(group) if i != selected_index]\n        examples.append(\n            CompletionExample(\n                task_id=f"{split}-{len(examples)}",\n                file_path=selected_path,\n                left_context=left,\n                target_code=target,\n                language=language,\n                related_files=related,\n            )\n        )\n    if len(examples) != count:\n        raise RuntimeError(f"Built only {len(examples)}/{count} examples after {attempts} attempts")\n    return examples\n\n\ndef chunk_related_files(example: CompletionExample, block_lines: int) -> list[Candidate]:\n    candidates: list[Candidate] = []\n    for path, content in example.related_files:\n        lines = [line for line in content.splitlines() if line.strip()][:5000]\n        for start in range(0, len(lines), block_lines):\n            block = lines[start : start + block_lines]\n            if not block:\n                continue\n            end = start + len(block) - 1\n            candidates.append(\n                Candidate(\n                    file_path=path,\n                    description=f"file path: {path}\\nlines: {start}-{end}",\n                    code="\\n".join(block),\n                    language=example.language,\n                )\n            )\n    return candidates\n\n\ndef lexical_tokens(text: str) -> list[str]:\n    return re.findall(r"[A-Za-z_][A-Za-z_0-9]*|\\d+|\\S", text.lower())\n\n\ndef bm25_candidates(example: CompletionExample, topk: int, block_lines: int) -> list[Candidate]:\n    blocks = chunk_related_files(example, block_lines)\n    if not blocks:\n        return []\n    corpus = [lexical_tokens(block.code) for block in blocks]\n    index = BM25Okapi(corpus)\n    query = "\\n".join(line for line in example.left_context.splitlines() if line.strip())[-8000:]\n    scores = index.get_scores(lexical_tokens(query))\n    order = np.argsort(scores)[::-1][:topk]\n    return [blocks[int(i)] for i in order]\n\n\ndef rlcoder_tokenize(text: str, tokenizer, max_length: int, is_query: bool) -> list[int]:\n    tokens = tokenizer.tokenize(text)\n    room = max_length - 4\n    tokens = tokens[-room:] if is_query else tokens[:room]\n    tokens = [tokenizer.cls_token, "<encoder-only>", tokenizer.sep_token] + tokens + [tokenizer.sep_token]\n    ids = tokenizer.convert_tokens_to_ids(tokens)\n    return ids + [tokenizer.pad_token_id] * (max_length - len(ids))\n\n\ndef mean_pool(last_hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    weights = mask.unsqueeze(-1).to(last_hidden.dtype)\n    return (last_hidden * weights).sum(1) / weights.sum(1).clamp_min(1)\n\n\n@torch.inference_mode()\ndef encode_texts(\n    model,\n    tokenizer,\n    texts: Sequence[str],\n    max_length: int,\n    batch_size: int,\n    device,\n    *,\n    is_query: bool = False,\n) -> torch.Tensor:\n    output: list[torch.Tensor] = []\n    for start in range(0, len(texts), batch_size):\n        ids = [\n            rlcoder_tokenize(text, tokenizer, max_length, is_query)\n            for text in texts[start : start + batch_size]\n        ]\n        input_ids = torch.tensor(ids, dtype=torch.long, device=device)\n        mask = input_ids.ne(tokenizer.pad_token_id)\n        hidden = model(input_ids=input_ids, attention_mask=mask).last_hidden_state\n        output.append(F.normalize(mean_pool(hidden, mask), p=2, dim=-1).cpu())\n    return torch.cat(output)\n\n\ndef build_fixed_candidate_pools(\n    examples: Sequence[CompletionExample], args, device\n) -> list[tuple[CompletionExample, list[Candidate]]]:\n    tokenizer = AutoTokenizer.from_pretrained(args.retriever_model)\n    model = AutoModel.from_pretrained(args.retriever_model).to(device).eval()\n    pools: list[tuple[CompletionExample, list[Candidate]]] = []\n    for example in tqdm(examples, desc="Freezing candidate pools"):\n        lexical = bm25_candidates(example, args.bm25_topk, args.block_lines)\n        if len(lexical) < args.candidate_count - 1:\n            continue\n        query = "\\n".join(\n            line for line in example.left_context.splitlines() if line.strip()\n        )[-8000:]\n        q = encode_texts(\n            model,\n            tokenizer,\n            [query],\n            args.query_length,\n            1,\n            device,\n            is_query=True,\n        )[0]\n        docs = encode_texts(\n            model,\n            tokenizer,\n            [candidate.render() for candidate in lexical],\n            args.candidate_length,\n            args.retriever_batch_size,\n            device,\n        )\n        scores = docs @ q\n        keep = torch.topk(scores, k=args.candidate_count - 1).indices.tolist()\n        selected = [lexical[i] for i in keep]\n        selected.append(\n            Candidate("", NULL_CONTEXT, "", example.language, is_null=True)\n        )\n        pools.append((example, selected))\n    del model\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return pools\n\n\ndef generator_prompt(example: CompletionExample, candidate: Candidate, tokenizer, args) -> list[int]:\n    cross_ids = tokenizer.encode(candidate.render(), add_special_tokens=False)[: args.generator_crossfile_length]\n    marker = "#" if example.language == "python" else "//"\n    path_ids = tokenizer.encode(\n        f"\\n\\n{marker} file path: {example.file_path}\\n\\n", add_special_tokens=False\n    )\n    allowed = max(args.generator_context_length - len(cross_ids) - len(path_ids) - 10, 1)\n    in_file = tokenizer.encode(example.left_context, add_special_tokens=False)[-allowed:]\n    return (cross_ids + path_ids + in_file)[-args.generator_context_length :]\n\n\n@torch.inference_mode()\ndef score_generator_losses(pools, args, device) -> list[dict]:\n    tokenizer = AutoTokenizer.from_pretrained(args.generator_model)\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token_id = tokenizer.eos_token_id\n    dtype = torch.float16 if device.type == "cuda" else torch.float32\n    model = AutoModelForCausalLM.from_pretrained(args.generator_model, torch_dtype=dtype).to(device).eval()\n\n    flat: list[tuple[int, CompletionExample, Candidate]] = []\n    for example_index, (example, candidates) in enumerate(pools):\n        flat.extend((example_index, example, candidate) for candidate in candidates)\n    losses = [0.0] * len(flat)\n    for start in tqdm(range(0, len(flat), args.generator_batch_size), desc="Freezing PPL rewards"):\n        batch = flat[start : start + args.generator_batch_size]\n        sequences: list[list[int]] = []\n        labels: list[list[int]] = []\n        for _, example, candidate in batch:\n            prompt_ids = generator_prompt(example, candidate, tokenizer, args)\n            target_ids = tokenizer.encode(example.target_code, add_special_tokens=False)[: args.target_length]\n            sequences.append(prompt_ids + target_ids)\n            labels.append([-100] * len(prompt_ids) + target_ids)\n        width = max(len(sequence) for sequence in sequences)\n        padded_ids, padded_labels, masks = [], [], []\n        for sequence, label in zip(sequences, labels):\n            padding = width - len(sequence)\n            padded_ids.append([tokenizer.pad_token_id] * padding + sequence)\n            padded_labels.append([-100] * padding + label)\n            masks.append([0] * padding + [1] * len(sequence))\n        input_ids = torch.tensor(padded_ids, dtype=torch.long, device=device)\n        label_tensor = torch.tensor(padded_labels, dtype=torch.long, device=device)\n        attention_mask = torch.tensor(masks, dtype=torch.long, device=device)\n        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1].float()\n        shifted = label_tensor[:, 1:]\n        token_loss = F.cross_entropy(\n            logits.reshape(-1, logits.shape[-1]),\n            shifted.reshape(-1),\n            reduction="none",\n            ignore_index=-100,\n        ).view(shifted.shape)\n        per_sample = token_loss.sum(-1) / shifted.ne(-100).sum(-1).clamp_min(1)\n        losses[start : start + len(batch)] = per_sample.cpu().tolist()\n\n    rows: list[dict] = []\n    cursor = 0\n    for split_and_pool in pools:\n        example, candidates = split_and_pool\n        candidate_losses = losses[cursor : cursor + len(candidates)]\n        cursor += len(candidates)\n        rows.append(\n            {\n                "task_id": example.task_id,\n                "query": "\\n".join(\n                    line for line in example.left_context.splitlines() if line.strip()\n                )[-8000:],\n                "candidates": [candidate.render() for candidate in candidates],\n                "candidate_meta": [asdict(candidate) for candidate in candidates],\n                "generator_losses": candidate_losses,\n                "label": int(np.argmin(candidate_losses)),\n            }\n        )\n    del model\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return rows\n\n\ndef write_jsonl(path: Path, rows: Iterable[dict]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        for row in rows:\n            handle.write(json.dumps(row, ensure_ascii=False) + "\\n")\n\n\ndef read_jsonl(path: Path) -> list[dict]:\n    with path.open(encoding="utf-8") as handle:\n        return [json.loads(line) for line in handle if line.strip()]\n\n\ndef prepare_reward_cache(args, device) -> None:\n    data_root = resolve_data_root(args.data_root)\n    train_groups, valid_groups = load_repository_groups(data_root)\n    splits = {\n        "train": make_examples(train_groups, args.train_examples, args.seed, "train"),\n        "valid": make_examples(valid_groups, args.valid_examples, args.seed + 1, "valid"),\n    }\n    cache_dir = Path(args.cache_dir)\n    manifest = {\n        "seed": args.seed,\n        "retriever_model": args.retriever_model,\n        "generator_model": args.generator_model,\n        "candidate_count": args.candidate_count,\n        "bm25_topk": args.bm25_topk,\n        "train_examples_requested": args.train_examples,\n        "valid_examples_requested": args.valid_examples,\n    }\n    for split, examples in splits.items():\n        pools = build_fixed_candidate_pools(examples, args, device)\n        if len(pools) != len(examples):\n            print(f"Warning: kept {len(pools)}/{len(examples)} {split} examples with enough candidates")\n        rows = score_generator_losses(pools, args, device)\n        write_jsonl(cache_dir / f"{split}.jsonl", rows)\n        manifest[f"{split}_examples_written"] = len(rows)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    (cache_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    print(f"Frozen reward cache written to {cache_dir.resolve()}")\n\n\nclass FrozenRewardDataset(Dataset):\n    def __init__(self, rows: Sequence[dict], tokenizer, query_length: int, candidate_length: int):\n        self.rows = rows\n        self.tokenizer = tokenizer\n        self.query_length = query_length\n        self.candidate_length = candidate_length\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def __getitem__(self, index: int):\n        row = self.rows[index]\n        query = rlcoder_tokenize(row["query"], self.tokenizer, self.query_length, True)\n        candidates = [\n            rlcoder_tokenize(text, self.tokenizer, self.candidate_length, False)\n            for text in row["candidates"]\n        ]\n        return {\n            "query_ids": torch.tensor(query, dtype=torch.long),\n            "candidate_ids": torch.tensor(candidates, dtype=torch.long),\n            "label": torch.tensor(row["label"], dtype=torch.long),\n            "generator_losses": torch.tensor(row["generator_losses"], dtype=torch.float),\n        }\n\n\nclass QIEPSMFidelity(nn.Module):\n    """Original uncompressed QiBERT encoding and raw product fidelity.\n\n    No learned projection, normalization, learned phase, clamp or log score.\n    Source: qiepsm/compression experiments/bert_base/qi bert base trained/qi-bert.py\n    and its utils/utils.py fidelity(). One qubit per encoder dimension.\n    """\n\n    def states(self, embeddings: torch.Tensor) -> torch.Tensor:\n        theta = torch.tanh(embeddings) * torch.pi / 2 + torch.pi / 2\n        # Retain the original phase expression (approximately pi).\n        phase = torch.ones_like(theta) * (2 * torch.pi - 1e-9) / 2\n        return torch.stack((torch.cos(theta / 2),\n                            torch.sin(theta / 2) * torch.exp(1j * phase)), dim=-1)\n\n    def pairwise_fidelity(self, query: torch.Tensor, docs: torch.Tensor) -> torch.Tensor:\n        q = self.states(query).unsqueeze(1)\n        d = self.states(docs)\n        overlap = (q.conj() * d).sum(dim=-1)\n        # Diagnostic only: sum logs before product underflow, using the same states.\n        self.last_log_fidelity = 2 * overlap.detach().abs().log().sum(dim=-1)\n        return overlap.prod(dim=-1).abs().square()\n\n    def pairwise_log_fidelity(self, query, docs):\n        q, d = self.states(query).unsqueeze(1), self.states(docs)\n        magnitude = (q.conj()*d).sum(-1).abs()\n        # Only guard exact/numerically unrepresentable zero, not a 1e-7 floor.\n        value = 2 * magnitude.clamp_min(torch.finfo(magnitude.dtype).tiny).log().sum(-1)\n        self.last_log_fidelity = value.detach()\n        return value\n\n\nclass QIEPSMLearnedPhase(QIEPSMFidelity):\n    """Uncompressed theta encoding with an input-dependent trainable phase."""\n    def __init__(self, hidden_size):\n        super().__init__()\n        self.phase_projection = nn.Linear(hidden_size, hidden_size)\n        nn.init.normal_(self.phase_projection.weight, std=0.01 / math.sqrt(hidden_size))\n        nn.init.zeros_(self.phase_projection.bias)\n\n    def states(self, embeddings):\n        theta = torch.tanh(embeddings) * torch.pi / 2 + torch.pi / 2\n        phase = torch.pi + torch.pi * torch.tanh(self.phase_projection(embeddings))\n        return torch.stack((torch.cos(theta / 2),\n                            torch.sin(theta / 2) * torch.exp(1j * phase)), -1)\n\n\nclass QuantumFidelityHead(nn.Module):\n    """QIEPSM-inspired product-state projection with stable log fidelity."""\n\n    def __init__(self, hidden_size: int, n_qubits: int, dropout: float):\n        super().__init__()\n        self.n_qubits = n_qubits\n        self.projection = nn.Sequential(\n            nn.LayerNorm(hidden_size),\n            nn.Dropout(dropout),\n            nn.Linear(hidden_size, 2 * n_qubits),\n        )\n\n    def states(self, embeddings: torch.Tensor) -> torch.Tensor:\n        angles = self.projection(embeddings)\n        theta_raw, phi_raw = angles.chunk(2, dim=-1)\n        theta = torch.sigmoid(theta_raw) * math.pi\n        phi = torch.tanh(phi_raw) * math.pi\n        amplitude_0 = torch.cos(theta / 2).to(torch.complex64)\n        amplitude_1 = torch.sin(theta / 2).to(torch.complex64) * torch.exp(1j * phi.float())\n        return torch.stack((amplitude_0, amplitude_1), dim=-1)\n\n    def pairwise_log_fidelity(self, query: torch.Tensor, docs: torch.Tensor) -> torch.Tensor:\n        q_state = self.states(query).unsqueeze(1)\n        d_state = self.states(docs)\n        overlap = (q_state.conj() * d_state).sum(dim=-1).abs().square()\n        # mean(log F_i) ranks like product(F_i), without numerical underflow.\n        return torch.log(overlap.clamp_min(1e-7)).mean(dim=-1)\n\n\nclass ControlledRetriever(nn.Module):\n    def __init__(self, model_name: str, scorer: str, n_qubits: int, dropout: float, temperature: float):\n        super().__init__()\n        self.encoder = AutoModel.from_pretrained(model_name)\n        self.scorer = scorer\n        self.temperature = temperature\n        if scorer == "qiepsm_learned_phase":\n            self.quantum_head = QIEPSMLearnedPhase(self.encoder.config.hidden_size)\n        elif scorer in ("qiepsm_fidelity", "qiepsm_log_fidelity"):\n            if scorer == "qiepsm_fidelity" and temperature != 20.:\n                raise ValueError(\'QIEPSM fidelity requires the original score scale 20.\')\n            self.quantum_head = QIEPSMFidelity()\n        elif scorer == "quantum_fidelity":\n            self.quantum_head = QuantumFidelityHead(\n                self.encoder.config.hidden_size, n_qubits, dropout\n            )\n        elif scorer != "cosine":\n            raise ValueError(f"Unknown scorer: {scorer}")\n\n    def embed(self, input_ids: torch.Tensor, pad_token_id: int) -> torch.Tensor:\n        mask = input_ids.ne(pad_token_id)\n        hidden = self.encoder(input_ids=input_ids, attention_mask=mask).last_hidden_state\n        return mean_pool(hidden, mask)\n\n    def forward(self, query_ids: torch.Tensor, candidate_ids: torch.Tensor, pad_token_id: int):\n        batch, candidate_count, length = candidate_ids.shape\n        query = self.embed(query_ids, pad_token_id)\n        docs = self.embed(candidate_ids.reshape(-1, length), pad_token_id).view(batch, candidate_count, -1)\n        if self.scorer == "cosine":\n            return F.cosine_similarity(query.unsqueeze(1), docs, dim=-1) * self.temperature\n        if self.scorer == "qiepsm_fidelity":\n            return self.quantum_head.pairwise_fidelity(query.float(), docs.float()) * self.temperature\n        return self.quantum_head.pairwise_log_fidelity(query, docs) * self.temperature\n\n\ndef ranking_metrics(logits: torch.Tensor, labels: torch.Tensor, generator_losses: torch.Tensor) -> dict:\n    order = logits.argsort(dim=-1, descending=True)\n    ranks = order.eq(labels.unsqueeze(1)).float().argmax(dim=1) + 1\n    selected = logits.argmax(dim=-1)\n    selected_loss = generator_losses.gather(1, selected.unsqueeze(1)).squeeze(1)\n    oracle_loss = generator_losses.min(dim=1).values\n    return {\n        "correct": selected.eq(labels).sum().item(),\n        "mrr_sum": (1.0 / ranks.float()).sum().item(),\n        "ndcg_sum": (1.0 / torch.log2(ranks.float() + 1)).sum().item(),\n        "regret_sum": (selected_loss - oracle_loss).sum().item(),\n        "count": labels.numel(),\n    }\n\n\n@torch.inference_mode()\ndef evaluate(model, loader, pad_token_id: int, device) -> dict:\n    model.eval()\n    totals = {"loss_sum": 0.0, "correct": 0, "mrr_sum": 0.0, "ndcg_sum": 0.0, "regret_sum": 0.0, "count": 0}\n    for batch in loader:\n        batch = {key: value.to(device) for key, value in batch.items()}\n        logits = model(batch["query_ids"], batch["candidate_ids"], pad_token_id)\n        loss = F.cross_entropy(logits, batch["label"], reduction="sum")\n        metrics = ranking_metrics(logits, batch["label"], batch["generator_losses"])\n        totals["loss_sum"] += loss.item()\n        for key, value in metrics.items():\n            totals[key] += value\n    count = max(totals.pop("count"), 1)\n    return {\n        "loss": totals["loss_sum"] / count,\n        "top1_accuracy": totals["correct"] / count,\n        "mrr": totals["mrr_sum"] / count,\n        "ndcg": totals["ndcg_sum"] / count,\n        "generator_loss_regret": totals["regret_sum"] / count,\n    }\n\n\ndef save_checkpoint(model, tokenizer, output_dir: Path, epoch: int, args, optimizer, scheduler) -> None:\n    checkpoint = output_dir / f"checkpoint-epoch-{epoch}"\n    checkpoint.mkdir(parents=True, exist_ok=True)\n    model.encoder.save_pretrained(checkpoint / "encoder")\n    tokenizer.save_pretrained(checkpoint / "encoder")\n    if model.scorer == "quantum_fidelity":\n        torch.save(model.quantum_head.state_dict(), checkpoint / "quantum_head.pt")\n    torch.save(\n        {\n            "epoch": epoch,\n            "optimizer": optimizer.state_dict(),\n            "scheduler": scheduler.state_dict(),\n        },\n        checkpoint / "training_state.pt",\n    )\n    config = {\n        "scorer": model.scorer,\n        "temperature": model.temperature,\n        "quantum_n_qubits": args.quantum_n_qubits,\n        "quantum_dropout": args.quantum_dropout,\n        "base_model": args.retriever_model,\n    }\n    (checkpoint / "experiment_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")\n\n\ndef train_retriever(args, device) -> None:\n    cache_dir = Path(args.cache_dir)\n    train_rows = read_jsonl(cache_dir / "train.jsonl")\n    valid_rows = read_jsonl(cache_dir / "valid.jsonl")\n    tokenizer = AutoTokenizer.from_pretrained(args.retriever_model)\n    train_dataset = FrozenRewardDataset(train_rows, tokenizer, args.query_length, args.candidate_length)\n    valid_dataset = FrozenRewardDataset(valid_rows, tokenizer, args.query_length, args.candidate_length)\n    generator = torch.Generator().manual_seed(args.seed)\n    train_loader = DataLoader(\n        train_dataset,\n        batch_size=args.train_batch_size,\n        shuffle=True,\n        generator=generator,\n        num_workers=args.num_workers,\n        pin_memory=torch.cuda.is_available(),\n    )\n    valid_loader = DataLoader(\n        valid_dataset,\n        batch_size=args.eval_batch_size,\n        shuffle=False,\n        num_workers=args.num_workers,\n        pin_memory=torch.cuda.is_available(),\n    )\n    model = ControlledRetriever(\n        args.retriever_model,\n        args.scorer,\n        args.quantum_n_qubits,\n        args.quantum_dropout,\n        args.temperature,\n    ).to(device)\n    optimizer = AdamW(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)\n    total_steps = max(len(train_loader) * args.epochs, 1)\n    scheduler = get_linear_schedule_with_warmup(\n        optimizer,\n        num_warmup_steps=int(total_steps * args.warmup_ratio),\n        num_training_steps=total_steps,\n    )\n    use_amp = args.fp16 and device.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)\n    output_dir = Path(args.output_dir) / args.scorer\n    output_dir.mkdir(parents=True, exist_ok=True)\n    metrics_path = output_dir / "metrics.jsonl"\n\n    initial = evaluate(model, valid_loader, tokenizer.pad_token_id, device)\n    print("Initial validation:", json.dumps(initial, indent=2))\n    with metrics_path.open("w", encoding="utf-8") as log:\n        log.write(json.dumps({"epoch": 0, "split": "valid", **initial}) + "\\n")\n\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        loss_sum = 0.0\n        seen = 0\n        started = time.time()\n        progress = tqdm(train_loader, desc=f"{args.scorer} epoch {epoch}/{args.epochs}")\n        for batch in progress:\n            query_ids = batch["query_ids"].to(device, non_blocking=True)\n            candidate_ids = batch["candidate_ids"].to(device, non_blocking=True)\n            labels = batch["label"].to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):\n                logits = model(query_ids, candidate_ids, tokenizer.pad_token_id)\n                loss = F.cross_entropy(logits.float(), labels)\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            loss_sum += loss.item() * labels.numel()\n            seen += labels.numel()\n            progress.set_postfix(loss=f"{loss_sum / max(seen, 1):.4f}")\n\n        train_record = {\n            "epoch": epoch,\n            "split": "train",\n            "loss": loss_sum / max(seen, 1),\n            "seconds": time.time() - started,\n        }\n        valid_record = {"epoch": epoch, "split": "valid", **evaluate(model, valid_loader, tokenizer.pad_token_id, device)}\n        print(json.dumps(train_record, indent=2))\n        print(json.dumps(valid_record, indent=2))\n        with metrics_path.open("a", encoding="utf-8") as log:\n            log.write(json.dumps(train_record) + "\\n")\n            log.write(json.dumps(valid_record) + "\\n")\n        save_checkpoint(model, tokenizer, output_dir, epoch, args, optimizer, scheduler)\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)\n    parser.add_argument("--stage", choices=("prepare", "train", "all"), default="all")\n    parser.add_argument("--scorer", choices=("cosine", "quantum_fidelity"), default="cosine")\n    parser.add_argument("--data-root", default="/kaggle/input/data4rlcoder")\n    parser.add_argument("--cache-dir", default="/kaggle/working/rlcoder_controlled/cache")\n    parser.add_argument("--output-dir", default="/kaggle/working/rlcoder_controlled/runs")\n    parser.add_argument("--retriever-model", default="microsoft/unixcoder-base")\n    parser.add_argument("--generator-model", default="deepseek-ai/deepseek-coder-1.3b-base")\n    parser.add_argument("--seed", type=int, default=123)\n    parser.add_argument("--train-examples", type=int, default=500)\n    parser.add_argument("--valid-examples", type=int, default=100)\n    parser.add_argument("--candidate-count", type=int, default=6)\n    parser.add_argument("--bm25-topk", type=int, default=50)\n    parser.add_argument("--block-lines", type=int, default=12)\n    parser.add_argument("--query-length", type=int, default=256)\n    parser.add_argument("--candidate-length", type=int, default=512)\n    parser.add_argument("--retriever-batch-size", type=int, default=32)\n    parser.add_argument("--generator-batch-size", type=int, default=2)\n    parser.add_argument("--generator-context-length", type=int, default=512)\n    parser.add_argument("--generator-crossfile-length", type=int, default=384)\n    parser.add_argument("--target-length", type=int, default=64)\n    parser.add_argument("--epochs", type=int, default=5)\n    parser.add_argument("--train-batch-size", type=int, default=4)\n    parser.add_argument("--eval-batch-size", type=int, default=8)\n    parser.add_argument("--learning-rate", type=float, default=5e-5)\n    parser.add_argument("--weight-decay", type=float, default=0.01)\n    parser.add_argument("--warmup-ratio", type=float, default=0.1)\n    parser.add_argument("--max-grad-norm", type=float, default=1.0)\n    parser.add_argument("--temperature", type=float, default=20.0)\n    parser.add_argument("--quantum-n-qubits", type=int, default=64)\n    parser.add_argument("--quantum-dropout", type=float, default=0.1)\n    parser.add_argument("--num-workers", type=int, default=2)\n    parser.add_argument("--fp16", action=argparse.BooleanOptionalAction, default=True)\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    seed_everything(args.seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    print(f"Device: {device}; stage={args.stage}; scorer={args.scorer}")\n    if args.stage in ("prepare", "all"):\n        prepare_reward_cache(args, device)\n    if args.stage in ("train", "all"):\n        train_retriever(args, device)\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ENGINE / "generation_eval" / "requirements.txt")], check=True)


In [ ]:
from datetime import datetime
OUTPUT = Path("/kaggle/working") / ("rlcoder_" + SCORER + "_" + LOSS + "_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
command = [sys.executable, "-u", str(ENGINE / "online_learned_phase.py"),
    "--data-root", str(DATA), "--output-dir", str(OUTPUT),
    "--samples-per-epoch", str(16 if SMOKE else SAMPLES_PER_EPOCH),
    "--epochs", str(1 if SMOKE else EPOCHS),
    "--valid-examples", str(4 if SMOKE else VALID_PER_LANGUAGE),
    "--scorer", SCORER, "--loss", LOSS, "--score-scale", str(SCORE_SCALE), "--seed", str(SEED),
    "--generator-batch-size", str(GENERATOR_BATCH_SIZE),
    "--encode-batch-size", str(ENCODE_BATCH_SIZE),
    "--train-batch-size", str(TRAIN_BATCH_SIZE),
    "--accumulation-steps", str(ACCUMULATION_STEPS),
    "--log-every-updates", str(LOG_EVERY_UPDATES),
    "--eval-epochs", *map(str, EVAL_EPOCHS),
    "--benchmark-limit", str(BENCHMARK_LIMIT)]
command += ["--max-runtime-minutes", str(MAX_RUNTIME_MINUTES)]
if RESUME:
    command += ["--resume", RESUME]
print("Running:", " ".join(command))
log_path = OUTPUT / "notebook-run.log"
OUTPUT.mkdir(parents=True, exist_ok=True)
with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    tail = []
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
        tail.append(line.rstrip())
        tail = tail[-80:]
    code = process.wait()
if code:
    failure = OUTPUT / "failure.txt"
    phase = OUTPUT / "phase.json"
    print("\n--- LAST 80 ENGINE LINES ---")
    print("\n".join(tail))
    if phase.exists():
        print("\n--- FAILED PHASE ---\n" + phase.read_text(encoding="utf-8"))
    if failure.exists():
        print("\n--- FULL TRACEBACK ---\n" + failure.read_text(encoding="utf-8"))
    raise RuntimeError(f"Training engine failed with exit code {code}; diagnostics: {OUTPUT}")
print("Outputs:", OUTPUT)
print("Resume checkpoint:", OUTPUT / "latest.pt")